# Revision v2: multi-seed XAI / adversarial-evasion IDS experiment (Kaggle)

This notebook carries out the revised experiment for *"A Multi-Seed Evaluation of Explanation Faithfulness
and Stability under Adversarial Evasion in IoT/IIoT Intrusion Detection"* (major revision).

**Before running**
1. *Add Input* → the **ToN-IoT network** dataset (`train_test_network.csv`) and the **Edge-IIoTset** dataset
   (`ML-EdgeIIoT-dataset.csv`). These are the same inputs used for the original run.
2. *Settings* → Accelerator **GPU T4 x1** (or P100). Internet **On** (only needed to `pip install lime`).
3. In the **SESSION** cell, choose `SESSION = "TEST"` first (~15 min). Check that it finishes and that the printed
   macro-F1 values look reasonable. Then make one notebook version for each of `"A"`, `"B"`, `"C1"` and `"C2"`.
4. Use **Save Version → Save & Run All (Commit)** for A/B/C so that each one runs in the background (≤12 h).
   Download `results_session_<X>.zip` from each version's *Output* tab and send all four zips back.

| Session | Content | Expected GPU time |
|---|---|---|
| A | ToN-IoT: 20 seeds × {MLP, MLP+FGSM-AT, MLP+PGD-AT, XGBoost}, all attacks, SHAP/LIME, repeated-call | see log |
| B | Edge-IIoTset: same as A | see log |
| C1 | Partition sensitivity: 2 extra stratified partitions × 5 seeds, both datasets | see log |
| C2 | Adversarial-training trade-off sweep, baseline ablation, multiclass attack-family analysis, memory/latency profiling | see log |
| C3 | Attack success at false-positive-rate-matched operating points (short) | see log |
| C4 | Original-pipeline replication path (feature list, scaling) + absolute-memory profiling (short) | see log |

Each run writes `runs/<dataset>_split<k>_seed<s>/done.json`. If a session is interrupted, attach the previous
version's output as an input and set `RESUME_FROM` so that the completed runs are skipped.

In [ ]:
import importlib.util, subprocess, sys
for pkg in ["lime"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import torch, shap, xgboost
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| shap", shap.__version__, "| xgboost", xgboost.__version__)

In [ ]:
import os; os.makedirs('/kaggle/working/iotxai', exist_ok=True)
open('/kaggle/working/iotxai/__init__.py','w').write('''"""iotxai: multi-seed evaluation of explanation faithfulness and stability under
adversarial evasion for tabular IoT/IIoT intrusion detection (revision v2)."""
__version__ = "2.0.0"
''')

In [ ]:
%%writefile /kaggle/working/iotxai/config.py
"""Experiment configuration for the revised (v2) multi-seed XAI/adversarial IDS study.

Every value used by the pipeline is defined here and written to ``config.json``
at the start of every run so that a result bundle is self-describing.
"""
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from pathlib import Path
import json

# Original five seeds are kept as the first five so the v1 run is a subset.
SEEDS_20 = [17, 29, 41, 53, 67, 83, 97, 101, 113, 127,
            131, 149, 163, 179, 191, 211, 223, 239, 251, 269]


@dataclass
class Config:
    # ------------------------------------------------------------------ I/O
    out_dir: str = "/kaggle/working/results_v2"
    work_dir: str = "/kaggle/working/cache_v2"
    input_root: str = "/kaggle/input"
    # Explicit file paths (leave "" for automatic discovery under input_root).
    ton_csv: str = ""
    edge_csv: str = ""

    # ---------------------------------------------------------- design
    datasets: list = field(default_factory=lambda: ["ton", "edge"])
    seeds: list = field(default_factory=lambda: list(SEEDS_20))
    split_ids: list = field(default_factory=lambda: [0])     # 0 = primary partition
    split_seed_base: int = 2026                                # split s uses 2026 + s
    max_rows_per_dataset: int | None = None                    # None = all rows after dedup
    dedup: bool = True                                         # de-duplicate before partitioning
    feature_policy: str = "v2"                                 # "v2" | "numeric_only" | "v1" (ablation)
    scaling: str = "logminmax"                                 # "logminmax" | "quantile_v1" (ablation)
    val_frac: float = 0.15
    test_frac: float = 0.15

    # ---------------------------------------------------------- models
    models: list = field(default_factory=lambda: ["mlp_clean", "mlp_fgsmat", "mlp_pgdat", "xgb"])
    hidden_dims: list = field(default_factory=lambda: [128, 64, 32])
    dropout: float = 0.2
    lr: float = 1e-3
    batch_size: int = 2048
    max_epochs: int = 30
    patience: int = 5
    # FGSM-AT reproduces the v1 control (eps=0.01, adversarial batch only).
    fgsmat_eps: float = 0.01
    fgsmat_lambda: float = 1.0
    # PGD-AT is the stronger defence requested by reviewers (mixed clean/adv loss).
    pgdat_eps: float = 0.03
    pgdat_steps: int = 5
    pgdat_lambda: float = 0.5
    xgb_n_estimators: int = 400
    xgb_max_depth: int = 6
    xgb_lr: float = 0.1
    xgb_subsample: float = 0.8
    xgb_colsample: float = 0.8

    # ---------------------------------------------------------- attacks
    epsilons: list = field(default_factory=lambda: [0.01, 0.03, 0.05])
    pgd_steps: int = 10
    attack_rows: int = 2000            # malicious test records, stratified by family
    bb_queries: int = 300              # score-based black-box attack budget
    # unconstrained = every input column (v1-style, incl. one-hot columns);
    # numeric = all numeric fields, both directions, categorical blocks frozen (closest to the v1 feature space);
    # constrained = mutability mask + increase-only + integer/relation repair.
    threat_models: list = field(default_factory=lambda: ["unconstrained", "numeric", "constrained"])
    # padding budgets beyond the l_inf grid, constrained threat model only (1.0 = up to the training maximum)
    constrained_extra_eps: list = field(default_factory=lambda: [0.1, 0.25, 0.5, 1.0])
    relation_min_support: float = 0.995  # data-derived invariants must hold in >=99.5% of train rows

    # ---------------------------------------------------------- explanations
    explain: bool = True
    explain_benign_rows: int = 300
    explain_malicious_rows: int = 300
    shap_background_rows: int = 100
    lime_num_samples: int = 1000
    lime_background_rows: int = 2000
    k_fracs: list = field(default_factory=lambda: [0.1, 0.2, 0.3, 0.4, 0.5])
    stability_ks: list = field(default_factory=lambda: [3, 5])
    marginal_draws: int = 16
    knn_neighbors: int = 8
    knn_pool_rows: int = 5000
    random_order_draws: int = 3
    # MLP views: PGD constrained at all eps + PGD unconstrained at max eps.
    # XGB views: black-box constrained/unconstrained at max eps.
    adv_views_reduced: bool = False    # True = only constrained@max-eps (split-sensitivity runs)
    repeat_seeds: int = 5              # repeated-call LIME/SHAP-background on the first N seeds
    repeat_rows: int = 100
    lime_repeats: int = 5
    shap_bg_repeats: int = 3

    # ---------------------------------------------------------- extras
    run_at_sweep: bool = False
    at_sweep_eps: list = field(default_factory=lambda: [0.01, 0.03, 0.05])
    at_sweep_lambda: list = field(default_factory=lambda: [0.5, 1.0])
    at_sweep_seeds: int = 5
    run_multiclass: bool = False
    multiclass_seeds: int = 5
    run_profiling: bool = False
    run_baseline_ablation: bool = False
    run_operating_point: bool = False
    run_v1_path: bool = False
    profile_rows: int = 100

    device: str = "auto"
    print_tables: bool = True          # print each run's result tables into the log (copy-paste fallback)

    def to_json(self, path: Path):
        Path(path).write_text(json.dumps(asdict(self), indent=2))


# --------------------------------------------------------------------------
# Dataset feature policies.  Columns not listed are auto-classified: numeric if
# >=95% of values parse as numbers, otherwise dropped (and logged).
# Roles of numeric features for the constrained threat model:
#   "up"   = mutable, attacker may only increase (padding, delay, extra packets)
#   "both" = mutable in both directions
#   anything not listed in MUTABLE = immutable (protocol semantics, server
#            responses, checksums, codes, one-hot categorical blocks).
# --------------------------------------------------------------------------
DATASET_SPECS = {
    "ton": {
        "label": "label",
        "family": "type",
        "benign_family": "normal",
        "file_patterns": ["train_test_network", "ton_iot", "network_dataset"],
        "drop": ["ts", "src_ip", "dst_ip", "src_port", "dst_port", "dns_query",
                 "ssl_subject", "ssl_issuer", "http_uri", "http_user_agent",
                 "http_referrer", "weird_name", "weird_addl"],
        "categorical": ["proto", "service", "conn_state", "dns_AA", "dns_RD", "dns_RA",
                        "dns_rejected", "ssl_version", "ssl_cipher", "ssl_resumed",
                        "ssl_established", "http_method", "http_version",
                        "http_orig_mime_types", "http_resp_mime_types", "weird_notice"],
        "integer": ["src_bytes", "dst_bytes", "missed_bytes", "src_pkts", "dst_pkts",
                    "src_ip_bytes", "dst_ip_bytes", "http_request_body_len",
                    "http_response_body_len", "http_trans_depth", "dns_qclass",
                    "dns_qtype", "dns_rcode", "http_status_code"],
        "mutable": {"duration": "up", "src_bytes": "up", "src_pkts": "up",
                    "src_ip_bytes": "up", "http_request_body_len": "up"},
        # Explicit protocol relations (raw units): lhs >= sum(coef * feature) holds
        # for valid flows. Each is kept only if it holds in the clean training data.
        "relations": [
            {"lhs": "src_ip_bytes", "rhs": {"src_bytes": 1.0, "src_pkts": 20.0},
             "why": "IP bytes include payload plus >=20-byte IPv4 header per packet"},
            {"lhs": "dst_ip_bytes", "rhs": {"dst_bytes": 1.0, "dst_pkts": 20.0},
             "why": "same invariant for responder direction"},
        ],
    },
    "edge": {
        "label": "Attack_label",
        "family": "Attack_type",
        "benign_family": "normal",
        "file_patterns": ["ml-edgeiiot-dataset", "ml_edgeiiot", "dnn-edgeiiot-dataset"],
        # Ferrag et al. (2022) recommended drops + capture-artefact identifiers.
        "drop": ["frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4",
                 "arp.dst.proto_ipv4", "http.file_data", "http.request.full_uri",
                 "icmp.transmit_timestamp", "http.request.uri.query", "tcp.options",
                 "tcp.payload", "tcp.srcport", "tcp.dstport", "udp.port", "mqtt.msg",
                 "udp.stream", "tcp.ack_raw", "mbtcp.trans_id", "mbtcp.unit_id",
                 "http.tls_port"],
        "categorical": ["http.request.method", "http.referer", "http.request.version",
                        "dns.qry.name.len", "mqtt.conack.flags", "mqtt.protoname",
                        "mqtt.topic"],
        "integer": ["tcp.len", "http.content_length", "mqtt.len", "tcp.ack", "tcp.seq",
                    "icmp.seq_le", "mqtt.msg_decoded_as", "mqtt.topic_len",
                    "mbtcp.len", "tcp.flags", "mqtt.hdrflags", "mqtt.msgtype"],
        "mutable": {"tcp.len": "up", "http.content_length": "up", "mqtt.len": "up"},
        "relations": [],
    },
}

ALL_MODELS = ["mlp_clean", "mlp_fgsmat", "mlp_pgdat", "xgb"]

# Exact feature lists of the original (v1) submission, from its archived metadata.
V1_FEATURES = {
    "ton": ["duration", "src_bytes", "dst_bytes", "missed_bytes", "src_pkts", "dst_pkts", "dns_qtype", "dns_rcode"],
    "edge": ["arp.opcode", "arp.hw.size", "icmp.checksum", "icmp.seq_le", "http.content_length", "http.response",
             "tcp.ack", "tcp.ack_raw", "tcp.checksum", "tcp.connection.fin", "tcp.connection.rst",
             "tcp.connection.syn", "tcp.connection.synack", "tcp.flags", "tcp.flags.ack", "tcp.len", "tcp.seq",
             "udp.stream", "dns.qry.name", "mqtt.hdrflags", "mqtt.len", "mqtt.msgtype"],
}


def session_config(session: str, **base) -> Config:
    """Presets for the reported run (one Kaggle notebook version per session)."""
    s = session.upper()
    if s == "TEST":
        return Config(**base, datasets=["ton", "edge"], seeds=[17], max_epochs=5, xgb_n_estimators=100,
                      explain_benign_rows=50, explain_malicious_rows=50, attack_rows=300, lime_num_samples=300,
                      bb_queries=100, repeat_seeds=1, repeat_rows=20, lime_repeats=2, shap_bg_repeats=2)
    if s == "A":
        return Config(**base, datasets=["ton"])
    if s == "B":
        return Config(**base, datasets=["edge"])
    if s == "C1":   # partition sensitivity: 2 extra stratified partitions x 5 seeds
        return Config(**base, datasets=["ton", "edge"], split_ids=[1, 2], seeds=SEEDS_20[:5],
                      adv_views_reduced=True, repeat_seeds=0)
    if s == "C2":   # AT trade-off sweep, multiclass, resource profiling
        return Config(**base, datasets=["ton", "edge"], seeds=SEEDS_20[:5], split_ids=[],
                      run_at_sweep=True, run_multiclass=True, run_profiling=True,
                      run_baseline_ablation=True)
    if s == "C4":   # v1 -> v2 baseline path (feature list, scaling) + absolute-memory profiling
        return Config(**base, datasets=["ton", "edge"], seeds=SEEDS_20[:5], split_ids=[],
                      run_v1_path=True, run_profiling=True)
    if s == "C3":   # attack success vs. FPR-matched operating points (short)
        return Config(**base, datasets=["ton", "edge"], seeds=SEEDS_20[:5], split_ids=[],
                      run_operating_point=True)
    raise ValueError(session)

In [ ]:
%%writefile /kaggle/working/iotxai/data.py
"""Data loading, auditing, de-duplication, partitioning and preprocessing.

Design rules
------------
* Raw-file SHA-256, row counts, duplicate counts and label conflicts are logged.
* De-duplication is performed on the model-visible feature vector + label BEFORE
  partitioning, so no identical record can appear in two partitions; feature
  vectors that occur with both labels are removed as irreducibly ambiguous.
* Everything learned from data (categorical levels, constant-column removal,
  scaling ranges, relation support) is fitted on the training partition only.
* Partition membership is saved as raw-file row numbers so the exact splits can
  be rebuilt from the public raw CSV plus this code.
"""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from .config import DATASET_SPECS, V1_FEATURES

MISSING_TOKENS = {"-", "", "nan", "none", "null", "?"}
MAX_LEVELS = 20


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def discover_file(cfg, name):
    explicit = cfg.ton_csv if name == "ton" else cfg.edge_csv
    if explicit:
        return Path(explicit)
    pats = DATASET_SPECS[name]["file_patterns"]
    cands = []
    for p in Path(cfg.input_root).rglob("*.csv"):
        low = p.name.lower()
        for rank, pat in enumerate(pats):
            if pat in low:
                cands.append((rank, p.stat().st_size, p))
    if not cands:
        raise FileNotFoundError(f"No CSV for {name} under {cfg.input_root}; set cfg.{name}_csv")
    cands.sort(key=lambda t: (t[0], t[1]))
    return cands[0][2]


def _norm_str(s: pd.Series) -> pd.Series:
    out = s.astype(str).str.strip()
    out = out.where(~out.str.lower().isin(MISSING_TOKENS), "__missing__")
    return out


def _canon_cat(s: pd.Series) -> pd.Series:
    """Categorical strings; numeric-looking tokens are canonicalised so that
    e.g. '0' and '0.0' (a known Edge-IIoTset export quirk) are one level."""
    st = _norm_str(s)
    num = pd.to_numeric(st, errors="coerce")
    canon = num.map(lambda v: f"{v:g}" if pd.notna(v) else None)
    return canon.where(num.notna(), st)


def load_and_audit(cfg, name, log):
    spec = DATASET_SPECS[name]
    path = discover_file(cfg, name)
    log(f"[{name}] reading {path}")
    drop = set(spec["drop"])
    keep_only = None
    if cfg.feature_policy == "v1":   # exact v1 feature list (ablation only)
        keep_only = set(V1_FEATURES[name]) | {spec["label"], spec["family"]}
        drop = set()
    if keep_only is not None:
        raw = pd.read_csv(path, usecols=lambda c: c.strip() in keep_only, low_memory=False)
    else:
        raw = None
    try:
        if raw is not None:
            raise StopIteration
        raw = pd.read_csv(path, usecols=lambda c: c.strip() not in drop, low_memory=False)
    except StopIteration:
        pass
    except Exception as exc:  # malformed quoting in payload-like columns
        log(f"[{name}] C parser failed ({exc}); retrying with python engine")
        raw = pd.read_csv(path, usecols=lambda c: c.strip() not in drop, engine="python",
                          on_bad_lines="skip")
    raw.columns = [c.strip() for c in raw.columns]
    raw = raw.reset_index(drop=True)
    raw["__row__"] = np.arange(len(raw))
    audit = {"dataset": name, "source_path": str(path), "sha256": sha256_file(path),
             "file_bytes": Path(path).stat().st_size, "rows_raw": int(len(raw)),
             "columns_loaded": [c for c in raw.columns if c != "__row__"],
             "columns_dropped_by_policy": sorted(drop)}

    y_raw = pd.to_numeric(raw[spec["label"]], errors="coerce")
    fam = raw[spec["family"]].astype(str).str.strip().str.lower()
    bad = y_raw.isna() | ~y_raw.isin([0, 1])
    audit["rows_missing_or_invalid_label"] = int(bad.sum())
    raw, y_raw, fam = raw[~bad], y_raw[~bad].astype(int), fam[~bad]

    # ---- classify columns
    feats_num, feats_cat, dropped_auto = [], [], {}
    for c in raw.columns:
        if c in (spec["label"], spec["family"], "__row__"):
            continue
        if c in spec["categorical"]:
            if cfg.feature_policy == "numeric_only":
                dropped_auto[c] = "categorical_excluded_by_numeric_only_ablation"
            else:
                feats_cat.append(c)
            continue
        s = raw[c]
        if pd.api.types.is_numeric_dtype(s):
            feats_num.append(c)
            continue
        st = _norm_str(s)
        nonmiss = st[st != "__missing__"]
        parsed = pd.to_numeric(nonmiss, errors="coerce")
        if len(nonmiss) == 0 or parsed.notna().mean() >= 0.95:
            feats_num.append(c)
        else:
            dropped_auto[c] = "non_numeric_not_in_categorical_policy"
    audit["dropped_auto"] = dropped_auto

    num = pd.DataFrame(index=raw.index)
    for c in feats_num:
        s = raw[c]
        if not pd.api.types.is_numeric_dtype(s):
            s = pd.to_numeric(_norm_str(s).replace("__missing__", np.nan), errors="coerce")
        num[c] = s.astype(float)
    audit["numeric_missing_counts"] = {c: int(num[c].isna().sum()) for c in feats_num if num[c].isna().any()}
    num = num.fillna(0.0)   # '-' / empty denote an absent protocol field
    cat = pd.DataFrame({c: _canon_cat(raw[c]) for c in feats_cat}, index=raw.index)

    # ---- duplicate audit and de-duplication on model-visible features + label
    key = pd.concat([num.round(9), cat], axis=1)
    audit["rows_exact_duplicate_loaded_columns"] = int(raw.drop(columns="__row__").duplicated().sum())
    hashes = pd.util.hash_pandas_object(key, index=False)
    frame = pd.DataFrame({"h": hashes.values, "y": y_raw.values}, index=raw.index)
    n_labels = frame.groupby("h")["y"].nunique()
    conflict_h = set(n_labels[n_labels > 1].index)
    is_conflict = frame["h"].isin(conflict_h)
    audit["rows_in_label_conflicting_groups"] = int(is_conflict.sum())
    keep = (~is_conflict) & (~frame.duplicated(subset=["h", "y"], keep="first"))
    if not cfg.dedup:   # ablation only: keep duplicates and conflicting records
        keep = pd.Series(True, index=frame.index)
    audit["rows_duplicate_feature_label"] = int((~is_conflict & frame.duplicated(subset=["h", "y"], keep="first")).sum())
    num, cat, y, fam = num[keep], cat[keep], y_raw[keep].values, fam[keep].values
    rows = raw.loc[keep, "__row__"].values
    audit["rows_after_dedup"] = int(len(rows))

    if cfg.max_rows_per_dataset and len(rows) > cfg.max_rows_per_dataset:
        rng = np.random.default_rng(cfg.split_seed_base)
        sel = []
        frac = cfg.max_rows_per_dataset / len(rows)
        for f in np.unique(fam):
            idx = np.where(fam == f)[0]
            k = max(1, int(round(len(idx) * frac)))
            sel.append(rng.choice(idx, size=min(k, len(idx)), replace=False))
        sel = np.sort(np.concatenate(sel))
        num, cat, y, fam, rows = num.iloc[sel], cat.iloc[sel], y[sel], fam[sel], rows[sel]
        audit["rows_after_cap"] = int(len(rows))

    fam_counts = pd.Series(fam).value_counts()
    audit["family_counts_after_dedup"] = {k: int(v) for k, v in fam_counts.items()}
    audit["positive_rate_after_dedup"] = float(np.mean(y))
    data = {"num": num.reset_index(drop=True), "cat": cat.reset_index(drop=True),
            "y": np.asarray(y, int), "family": np.asarray(fam, object), "row": np.asarray(rows, int)}
    return data, audit


def make_split(data, split_id, cfg):
    seed = cfg.split_seed_base + split_id
    fam = data["family"]
    strat = fam.copy()
    vc = pd.Series(fam).value_counts()
    rare = set(vc[vc < 10].index)
    strat = np.array([("rare_" + str(yy)) if f in rare else f for f, yy in zip(fam, data["y"])], dtype=object)
    idx = np.arange(len(fam))
    hold = cfg.val_frac + cfg.test_frac
    tr, rest = train_test_split(idx, test_size=hold, random_state=seed, stratify=strat)
    va, te = train_test_split(rest, test_size=cfg.test_frac / hold, random_state=seed,
                              stratify=strat[rest])
    return {"train": np.sort(tr), "val": np.sort(va), "test": np.sort(te), "seed": seed}


class Preprocessor:
    """Train-fitted transform: numeric -> log1p (if non-negative) -> min-max to [0,1];
    categorical -> one-hot over the <=20 most frequent training levels (+__other__)."""

    def fit(self, data, train_idx, spec, cfg, log):
        num = data["num"].iloc[train_idx]
        self.num_cols, self.params, self.const_dropped = [], {}, []
        for c in num.columns:
            v = num[c].values
            if np.nanmax(v) == np.nanmin(v):
                self.const_dropped.append(c)
                continue
            if getattr(cfg, "scaling", "logminmax") == "quantile_v1":
                # v1 scaling: train 1st/99th percentiles, linear, clipped; zero span -> removed
                uselog = False
                lo, hi = float(np.quantile(v, 0.01)), float(np.quantile(v, 0.99))
                if hi <= lo:
                    self.const_dropped.append(c)
                    continue
            else:
                uselog = bool(np.nanmin(v) >= 0)
                t = np.log1p(v) if uselog else v
                lo, hi = float(np.min(t)), float(np.max(t))
            self.params[c] = {"log": uselog, "lo": lo, "hi": hi,
                              "raw_min": float(np.min(v)), "raw_max": float(np.max(v)),
                              "integer": bool(c in spec["integer"] or np.all(np.mod(v, 1) == 0))}
            self.num_cols.append(c)
        self.cat_levels = {}
        cat = data["cat"].iloc[train_idx]
        for c in cat.columns:
            vc = cat[c].value_counts()
            if len(vc) <= 1:
                self.const_dropped.append(c)
                continue
            levels = list(vc.index[:MAX_LEVELS])
            if len(vc) > MAX_LEVELS:
                levels.append("__other__")
            self.cat_levels[c] = levels
        # column layout and groups
        self.columns, self.groups, self.col_kind = [], [], []
        for c in self.num_cols:
            self.groups.append({"name": c, "cols": [len(self.columns)], "kind": "num"})
            self.columns.append(c); self.col_kind.append("num")
        for c, levels in self.cat_levels.items():
            start = len(self.columns)
            for lv in levels:
                self.columns.append(f"{c}={lv}"); self.col_kind.append("cat")
            self.groups.append({"name": c, "cols": list(range(start, len(self.columns))), "kind": "cat"})
        # mutability
        self.mutable_dir = np.zeros(len(self.columns), dtype=np.int8)  # 0 immutable, 1 up, 2 both
        for c, d in spec["mutable"].items():
            if c in self.num_cols:
                self.mutable_dir[self.columns.index(c)] = 1 if d == "up" else 2
        # relations kept only when supported by the clean training data
        self.relations = []
        for rel in spec["relations"]:
            need = [rel["lhs"]] + list(rel["rhs"])
            if not all(n in data["num"].columns for n in need):
                continue
            lhs = num[rel["lhs"]].values
            rhs = sum(coef * num[f].values for f, coef in rel["rhs"].items())
            support = float(np.mean(lhs >= rhs - 1e-6))
            r = dict(rel, support=support, kept=support >= cfg.relation_min_support)
            self.relations.append(r)
            log(f"relation {rel['lhs']} >= {rel['rhs']}: support={support:.4f} kept={r['kept']}")
        return self

    def transform(self, data, idx):
        n = len(idx)
        Z = np.zeros((n, len(self.columns)), dtype=np.float32)
        num = data["num"].iloc[idx]
        for j, c in enumerate(self.num_cols):
            p = self.params[c]
            v = num[c].values.astype(float)
            t = np.log1p(np.maximum(v, 0)) if p["log"] else v
            Z[:, j] = np.clip((t - p["lo"]) / (p["hi"] - p["lo"]), 0, 1)
        cat = data["cat"].iloc[idx]
        for g in self.groups:
            if g["kind"] != "cat":
                continue
            c = g["name"]; levels = self.cat_levels[c]
            v = cat[c].values
            other = "__other__" in levels
            for k, lv in enumerate(levels):
                if lv == "__other__":
                    Z[:, g["cols"][k]] = ~np.isin(v, levels)
                else:
                    Z[:, g["cols"][k]] = (v == lv)
            if not other:  # unseen level -> all-zero block (documented)
                pass
        return Z

    # ---- raw-unit helpers for constraint repair/validity
    def to_raw(self, Z, j):
        c = self.columns[j]; p = self.params[c]
        t = Z[:, j] * (p["hi"] - p["lo"]) + p["lo"]
        return np.expm1(t) if p["log"] else t

    def from_raw(self, v, j):
        c = self.columns[j]; p = self.params[c]
        t = np.log1p(np.maximum(v, 0)) if p["log"] else v
        return np.clip((t - p["lo"]) / (p["hi"] - p["lo"]), 0, 1)

    def meta(self):
        return {"columns": self.columns, "groups": self.groups, "col_kind": self.col_kind,
                "num_params": self.params, "cat_levels": self.cat_levels,
                "constant_dropped": self.const_dropped,
                "mutable": {self.columns[j]: int(d) for j, d in enumerate(self.mutable_dir) if d},
                "relations": self.relations}


def stratified_pick(idx, fam, n, rng):
    """Pick n indices from idx proportionally to family frequency (>=1 per family)."""
    idx = np.asarray(idx)
    if n >= len(idx):
        return rng.permutation(idx)
    fams = pd.Series(fam[idx])
    vc = fams.value_counts()
    alloc = np.maximum(1, np.floor(vc / vc.sum() * n)).astype(int)
    while alloc.sum() > n:
        alloc[alloc.idxmax()] -= 1
    while alloc.sum() < n:
        spare = (vc - alloc)[(vc - alloc) > 0]
        alloc[spare.idxmax()] += 1
    out = []
    for f, k in alloc.items():
        pool = idx[fams.values == f]
        out.append(rng.choice(pool, size=min(k, len(pool)), replace=False))
    return rng.permutation(np.concatenate(out))


def prepare(cfg, name, split_id, work_dir: Path, out_dir: Path, log, data_cache={}):
    """Return a dict with arrays + preprocessor for (dataset, split). Cached on disk."""
    work_dir.mkdir(parents=True, exist_ok=True)
    sc = getattr(cfg, "scaling", "logminmax")
    tag = "" if (cfg.dedup and cfg.feature_policy == "v2" and sc == "logminmax") else \
        f"_{cfg.feature_policy}_dedup{int(cfg.dedup)}" + ("" if sc == "logminmax" else f"_{sc}")
    cache = work_dir / f"{name}_split{split_id}{tag}.npz"
    meta_path = work_dir / f"{name}_split{split_id}{tag}_prep.pkl"
    import pickle
    if cache.exists() and meta_path.exists():
        z = dict(np.load(cache, allow_pickle=True))
        prep = pickle.loads(meta_path.read_bytes())
        return z, prep
    ckey = (name, cfg.feature_policy, cfg.dedup)   # scaling is applied after loading
    if ckey not in data_cache:
        data_cache[ckey] = load_and_audit(cfg, name, log)
    data, audit = data_cache[ckey]
    spec = DATASET_SPECS[name]
    sp = make_split(data, split_id, cfg)
    prep = Preprocessor().fit(data, sp["train"], spec, cfg, log)
    out = {}
    for part in ["train", "val", "test"]:
        out[f"X_{part}"] = prep.transform(data, sp[part])
        out[f"y_{part}"] = data["y"][sp[part]]
        out[f"fam_{part}"] = data["family"][sp[part]].astype(str)
        out[f"row_{part}"] = data["row"][sp[part]]
    # cross-partition duplicate verification on the transformed feature vectors
    def hv(X):
        return set(pd.util.hash_pandas_object(pd.DataFrame(X), index=False).values)
    htr, hva, hte = hv(out["X_train"]), hv(out["X_val"]), hv(out["X_test"])
    audit_split = {"split_id": split_id, "split_seed": sp["seed"],
                   "n_train": len(sp["train"]), "n_val": len(sp["val"]), "n_test": len(sp["test"]),
                   "train_test_shared_vectors": len(htr & hte), "train_val_shared_vectors": len(htr & hva),
                   "val_test_shared_vectors": len(hva & hte),
                   "n_model_columns": len(prep.columns), "n_feature_groups": len(prep.groups),
                   "train_positive_rate": float(out["y_train"].mean()),
                   "test_positive_rate": float(out["y_test"].mean())}
    # NB: shared *transformed* vectors can be >0 because clipping / top-20 level
    # bucketing maps distinct raw records to the same vector; raw duplicates are 0.
    rng = np.random.default_rng(sp["seed"])
    yt, ft = out["y_test"], out["fam_test"]
    mal = np.where(yt == 1)[0]; ben = np.where(yt == 0)[0]
    exp_mal = stratified_pick(mal, ft, cfg.explain_malicious_rows, rng)
    rest = np.setdiff1d(mal, exp_mal)
    extra = stratified_pick(rest, ft, max(0, cfg.attack_rows - len(exp_mal)), rng) if len(rest) else np.array([], int)
    out["attack_idx"] = np.concatenate([exp_mal, extra]).astype(int)        # first 300 = explained
    out["explain_benign_idx"] = rng.choice(ben, size=min(cfg.explain_benign_rows, len(ben)), replace=False)
    out["explain_mal_idx"] = exp_mal.astype(int)
    out["shap_bg_idx"] = rng.choice(len(out["y_train"]), size=cfg.shap_background_rows, replace=False)
    out["lime_bg_idx"] = rng.choice(len(out["y_train"]), size=min(cfg.lime_background_rows, len(out["y_train"])), replace=False)
    out["knn_pool_idx"] = rng.choice(len(out["y_train"]), size=min(cfg.knn_pool_rows, len(out["y_train"])), replace=False)
    np.savez_compressed(cache, **out)
    meta_path.write_bytes(pickle.dumps(prep))
    # public artefacts
    d = out_dir / ("data_audit" if not tag else f"data_audit_ablation{tag}"); d.mkdir(parents=True, exist_ok=True)
    (d / f"audit_{name}.json").write_text(json.dumps(audit, indent=2, default=str))
    (d / f"split_{name}_split{split_id}.json").write_text(json.dumps(audit_split, indent=2))
    (d / f"preprocessor_{name}_split{split_id}.json").write_text(json.dumps(prep.meta(), indent=2, default=str))
    parts = []
    for part in ["train", "val", "test"]:
        parts.append(pd.DataFrame({"raw_row": out[f"row_{part}"], "partition": part}))
    pd.concat(parts).sort_values("raw_row").to_csv(d / f"partition_{name}_split{split_id}.csv.gz", index=False)
    pd.DataFrame({"test_pos": out["attack_idx"], "raw_row": out["row_test"][out["attack_idx"]],
                  "explained": np.arange(len(out["attack_idx"])) < len(exp_mal)}).to_csv(
        d / f"attack_subset_{name}_split{split_id}.csv", index=False)
    pd.DataFrame({"test_pos": out["explain_benign_idx"], "raw_row": out["row_test"][out["explain_benign_idx"]]}).to_csv(
        d / f"explain_benign_{name}_split{split_id}.csv", index=False)
    log(f"[{name}] split {split_id}: {audit_split}")
    return out, prep

In [ ]:
%%writefile /kaggle/working/iotxai/models.py
"""Detectors (MLP family + gradient-boosted trees), training conditions,
validation-based threshold selection, calibration and predictive metrics."""
from __future__ import annotations

import random
import time

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, precision_score, recall_score,
                             roc_auc_score, brier_score_loss)


def resolve_device(pref="auto"):
    if pref == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(pref)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class MLP(nn.Module):
    def __init__(self, d_in, hidden, dropout, n_out=1):
        super().__init__()
        layers, d = [], d_in
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers.append(nn.Linear(d, n_out))
        self.net = nn.Sequential(*layers)
        self.n_out = n_out

    def forward(self, x):
        out = self.net(x)
        return out.squeeze(-1) if self.n_out == 1 else out


class Detector:
    """Uniform wrapper. ``predict(X)`` returns P(malicious) as float64 numpy."""

    def __init__(self, kind, obj, device, temperature=1.0):
        self.kind, self.obj, self.device = kind, obj, device
        self.temperature = temperature
        self.threshold = 0.5

    def logits(self, X, batch=65536):
        assert self.kind == "mlp"
        self.obj.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(X), batch):
                xb = torch.as_tensor(np.asarray(X[i:i + batch], np.float32), device=self.device)
                out.append(self.obj(xb).float().cpu().numpy())
        return np.concatenate(out) if out else np.zeros(0)

    def predict(self, X):
        X = np.asarray(X, np.float32)
        if len(X) == 0:
            return np.zeros(0)
        if self.kind == "mlp":
            return 1.0 / (1.0 + np.exp(-self.logits(X).astype(np.float64)))
        return self.obj.predict_proba(X)[:, 1].astype(np.float64)

    def predict_calibrated(self, X):
        if self.kind != "mlp":
            return self.predict(X)
        return 1.0 / (1.0 + np.exp(-self.logits(X).astype(np.float64) / self.temperature))


# ----------------------------------------------------------------- MLP training
def _perturb_unconstrained(model, xb, yb, wfn, eps, steps):
    """Loss-ascent perturbation for adversarial training (all features, l_inf, [0,1])."""
    model.eval()
    if steps <= 1:
        x = xb.clone().requires_grad_(True)
        loss = wfn(model(x), yb)
        g = torch.autograd.grad(loss, x)[0]
        adv = (xb + eps * g.sign()).clamp(0, 1)
    else:
        alpha = 2.5 * eps / steps
        adv = (xb + torch.empty_like(xb).uniform_(-eps, eps)).clamp(0, 1)
        for _ in range(steps):
            adv.requires_grad_(True)
            loss = wfn(model(adv), yb)
            g = torch.autograd.grad(loss, adv)[0]
            adv = adv.detach() + alpha * g.sign()
            adv = torch.max(torch.min(adv, xb + eps), xb - eps).clamp(0, 1)
    model.train()
    return adv.detach()


def train_mlp(X_tr, y_tr, X_va, y_va, seed, cfg, device, mode="clean", eps=0.0, lam=1.0,
              steps=1, n_classes=1, log=print):
    """mode: clean | at.  For AT the minibatch loss is lam*L(adv) + (1-lam)*L(clean)."""
    set_seed(seed)
    d = X_tr.shape[1]
    model = MLP(d, cfg.hidden_dims, cfg.dropout, n_out=n_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    if n_classes == 1:
        pos = float(np.mean(y_tr)); w = torch.tensor([0.5 / (1 - pos), 0.5 / pos], device=device)
        bce = nn.BCEWithLogitsLoss(reduction="none")
        def wfn(logit, yb):
            return (bce(logit, yb.float()) * w[yb.long()]).mean()
    else:
        counts = np.bincount(y_tr, minlength=n_classes).astype(float)
        cw = torch.tensor(len(y_tr) / (n_classes * np.maximum(counts, 1)), dtype=torch.float32, device=device)
        ce = nn.CrossEntropyLoss(weight=cw)
        def wfn(logit, yb):
            return ce(logit, yb.long())
    Xt = torch.as_tensor(X_tr, dtype=torch.float32)
    yt = torch.as_tensor(y_tr, dtype=torch.long)
    g = torch.Generator().manual_seed(seed)
    best, best_state, bad, hist = -1, None, 0, []
    t0 = time.time()
    for ep in range(cfg.max_epochs):
        model.train()
        perm = torch.randperm(len(Xt), generator=g)
        tot = 0.0
        for i in range(0, len(perm), cfg.batch_size):
            b = perm[i:i + cfg.batch_size]
            xb, yb = Xt[b].to(device), yt[b].to(device)
            if mode == "at":
                adv = _perturb_unconstrained(model, xb, yb, wfn, eps, steps)
                loss = lam * wfn(model(adv), yb)
                if lam < 1.0:
                    loss = loss + (1 - lam) * wfn(model(xb), yb)
            else:
                loss = wfn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += float(loss.detach()) * len(b)
        det = Detector("mlp", model, device)
        if n_classes == 1:
            pv = det.predict(X_va)
            score = f1_score(y_va, (pv >= 0.5).astype(int), average="macro", zero_division=0)
        else:
            pv = det.logits(X_va).argmax(1)
            score = f1_score(y_va, pv, average="macro", zero_division=0)
        hist.append({"epoch": ep + 1, "train_loss": tot / len(perm), "val_macro_f1": score})
        if score > best + 1e-5:
            best, bad = score, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg.patience:
                break
    model.load_state_dict(best_state); model.eval()
    return model, hist, time.time() - t0


def fit_temperature(det, X_va, y_va):
    """Temperature scaling on validation logits (does not change ranking)."""
    z = torch.as_tensor(det.logits(X_va), dtype=torch.float64)
    y = torch.as_tensor(y_va, dtype=torch.float64)
    logT = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
    bce = nn.BCEWithLogitsLoss()
    def closure():
        opt.zero_grad(); loss = bce(z / logT.exp(), y); loss.backward(); return loss
    opt.step(closure)
    return float(logT.detach().exp())


def train_xgb(X_tr, y_tr, X_va, y_va, seed, cfg, device, n_classes=1):
    import xgboost as xgb
    params = dict(n_estimators=cfg.xgb_n_estimators, max_depth=cfg.xgb_max_depth,
                  learning_rate=cfg.xgb_lr, subsample=cfg.xgb_subsample,
                  colsample_bytree=cfg.xgb_colsample, tree_method="hist", random_state=seed,
                  early_stopping_rounds=30, n_jobs=-1,
                  device="cuda" if device.type == "cuda" else "cpu")
    if n_classes == 1:
        pos = float(np.mean(y_tr))
        params.update(objective="binary:logistic", eval_metric="logloss",
                      scale_pos_weight=(1 - pos) / pos)
    else:
        params.update(objective="multi:softprob", eval_metric="mlogloss", num_class=n_classes)
    t0 = time.time()
    try:
        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    except Exception as exc:  # older xgboost without the `device` argument / no GPU build
        print("xgboost GPU/device fallback:", exc)
        params.pop("device", None)
        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    try:  # predictions/attacks/TreeSHAP on CPU for portability
        clf.set_params(device="cpu")
    except Exception:
        pass
    return clf, time.time() - t0


# ----------------------------------------------------------------- thresholds & metrics
def tune_threshold(y, p):
    """Validation threshold maximising macro-F1 (searched over <=2000 quantiles)."""
    qs = np.unique(np.quantile(p, np.linspace(0, 1, 2001)))
    best_t, best_f = 0.5, -1
    for t in qs:
        f = f1_score(y, (p >= t).astype(int), average="macro", zero_division=0)
        if f > best_f:
            best_t, best_f = float(t), f
    return best_t, best_f


def ece_score(y, p, bins=15):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    e = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)


def binary_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"threshold": thr, "accuracy": accuracy_score(y, pred),
            "balanced_accuracy": balanced_accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
            "mal_precision": precision_score(y, pred, zero_division=0),
            "mal_recall": recall_score(y, pred, zero_division=0),
            "mal_f1": f1_score(y, pred, zero_division=0),
            "ben_precision": tn / max(tn + fn, 1), "ben_recall": tn / max(tn + fp, 1),
            "fpr": fp / max(fp + tn, 1), "fnr": fn / max(fn + tp, 1),
            "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
            "pr_auc": average_precision_score(y, p), "roc_auc": roc_auc_score(y, p),
            "brier": brier_score_loss(y, p), "ece": ece_score(y, p)}


def family_recall(y, fam, p, thr):
    pred = (p >= thr).astype(int)
    rows = []
    for f in np.unique(fam):
        m = fam == f
        target = 1 if y[m].mean() > 0.5 else 0
        rows.append({"family": f, "n": int(m.sum()), "correct_rate": float(np.mean(pred[m] == target))})
    return rows

In [ ]:
%%writefile /kaggle/working/iotxai/attacks.py
"""Evasion attacks under two threat models and a domain-validity checker.

unconstrained : every model input may move by <= eps (l_inf) inside [0,1]
                (the v1 threat model, kept for continuity).
constrained   : only attacker-controllable flow features move (mutability mask),
                'up' features may only increase (padding / delay / extra packets),
                one-hot protocol blocks are frozen, and after the gradient search
                the record is repaired in raw units: integer rounding (ceil for
                increase-only features) and data-supported protocol relations
                (e.g. IP bytes >= payload bytes + 20 x packets).  Success is
                evaluated on the repaired record.
"""
from __future__ import annotations

import numpy as np
import torch


def masks(prep, threat, d):
    if threat == "unconstrained":
        return np.ones(d, bool), np.zeros(d, bool)
    if threat == "numeric":
        return np.array([k == "num" for k in prep.col_kind]), np.zeros(d, bool)
    mut = prep.mutable_dir > 0
    up = prep.mutable_dir == 1
    return mut, up


def _project(xa, x0, eps, mut, up):
    delta = torch.clamp(xa - x0, -eps, eps)
    delta = torch.where(up, delta.clamp(min=0), delta)
    delta = delta * mut
    return (x0 + delta).clamp(0, 1)


def pgd(det, X, eps, prep, threat, steps=10, seed=0, fgsm=False, target_logit_sign=1.0):
    """Minimise the malicious logit (binary) inside the feasible set."""
    model, device = det.obj, det.device
    model.eval()
    d = X.shape[1]
    mut_np, up_np = masks(prep, threat, d)
    mut = torch.as_tensor(mut_np, device=device, dtype=torch.float32)
    up = torch.as_tensor(up_np, device=device)
    out = []
    g = torch.Generator(device="cpu").manual_seed(int(seed))
    for i in range(0, len(X), 8192):
        x0 = torch.as_tensor(np.asarray(X[i:i + 8192], np.float32), device=device)
        if fgsm:
            xa = x0.clone().requires_grad_(True)
            grad = torch.autograd.grad(model(xa).sum(), xa)[0]
            xa = _project(x0 - eps * grad.sign(), x0, eps, mut, up)
        else:
            alpha = eps / max(1, steps) * 2.5
            noise = (torch.rand(x0.shape, generator=g) * 2 - 1).to(device) * eps
            xa = _project(x0 + noise, x0, eps, mut, up)
            for _ in range(steps):
                xa.requires_grad_(True)
                grad = torch.autograd.grad(model(xa).sum(), xa)[0]
                xa = _project(xa.detach() - alpha * grad.sign(), x0, eps, mut, up)
        out.append(xa.detach().cpu().numpy())
    Xa = np.concatenate(out).astype(np.float32)
    if threat == "constrained":
        Xa = repair(prep, Xa, np.asarray(X, np.float32))
    return Xa


def repair(prep, Xa, X0):
    """Raw-unit repair: integer rounding + data-supported relations. Only mutable
    columns are ever modified; if a relation cannot be restored by raising a
    mutable left-hand side, the offending mutable features are reverted."""
    Xa = Xa.copy()
    mut_cols = np.where(prep.mutable_dir > 0)[0]
    raw = {j: prep.to_raw(Xa, j) for j in mut_cols}
    raw0 = {j: prep.to_raw(X0, j) for j in mut_cols}
    for j in mut_cols:
        p = prep.params[prep.columns[j]]
        if p["integer"]:
            # floor keeps an increase-only change inside the eps budget
            raw[j] = np.floor(raw[j] + 1e-6) if prep.mutable_dir[j] == 1 else np.round(raw[j])
            raw[j] = np.minimum(raw[j], p["raw_max"])
        if prep.mutable_dir[j] == 1:
            raw[j] = np.maximum(raw[j], raw0[j])
    cols = prep.columns
    def val(name, Xsrc):
        j = cols.index(name)
        return raw[j] if j in raw else prep.to_raw(Xsrc, j)
    for rel in prep.relations:
        if not rel["kept"]:
            continue
        lhs = rel["lhs"]; jl = cols.index(lhs)
        rhs_val = sum(c * val(f, Xa) for f, c in rel["rhs"].items())
        rhs_val0 = sum(c * prep.to_raw(X0, cols.index(f)) for f, c in rel["rhs"].items())
        lhs0 = prep.to_raw(X0, jl)
        held0 = lhs0 >= rhs_val0 - 1e-6
        cur_l = val(lhs, Xa)
        viol = held0 & (cur_l < rhs_val - 1e-6)
        if not viol.any():
            continue
        if jl in raw:
            cap = prep.params[lhs]["raw_max"]
            new = np.where(viol, np.minimum(np.ceil(rhs_val), cap), raw[jl])
            raw[jl] = new
            still = viol & (raw[jl] < rhs_val - 1e-6)
        else:
            still = viol
        if still.any():   # revert the mutable rhs features for these rows
            for f in rel["rhs"]:
                jf = cols.index(f)
                if jf in raw:
                    raw[jf] = np.where(still, raw0[jf], raw[jf])
    for j in mut_cols:
        Xa[:, j] = prep.from_raw(raw[j], j)
    return Xa


def validity(prep, Xa, X0, tol=1e-4):
    """Per-record validity flags of an adversarial record w.r.t. the domain."""
    n, d = Xa.shape
    res = {}
    # one-hot integrity
    ok_cat = np.ones(n, bool)
    for g in prep.groups:
        if g["kind"] != "cat":
            continue
        B = Xa[:, g["cols"]]
        binary = np.all((np.abs(B) < tol) | (np.abs(B - 1) < tol), axis=1)
        ok_cat &= binary & (B.round().sum(1) <= 1)
    res["onehot_valid"] = ok_cat
    # integer features in raw units
    ok_int = np.ones(n, bool)
    for j, c in enumerate(prep.columns):
        if prep.col_kind[j] == "num" and prep.params[c]["integer"]:
            changed = np.abs(Xa[:, j] - X0[:, j]) > 1e-7
            v = prep.to_raw(Xa, j)
            ok_int &= ~changed | (np.abs(v - np.round(v)) <= 1e-3 * np.maximum(1, np.abs(v)))
    res["integer_valid"] = ok_int
    ok_rel = np.ones(n, bool)
    cols = prep.columns
    for rel in prep.relations:
        if not rel["kept"]:
            continue
        r = lambda Xs: (prep.to_raw(Xs, cols.index(rel["lhs"])),
                        sum(c * prep.to_raw(Xs, cols.index(f)) for f, c in rel["rhs"].items()))
        l0, r0 = r(X0); l1, r1 = r(Xa)
        tolr = 1e-3 * np.maximum(1, np.abs(r1))
        ok_rel &= ~(l0 >= r0 - 1e-6) | (l1 >= r1 - tolr)
    res["relation_valid"] = ok_rel
    changed = np.abs(Xa - X0) > 1e-6
    res["immutable_untouched"] = ~np.any(changed[:, prep.mutable_dir == 0], axis=1)
    up = prep.mutable_dir == 1
    res["direction_valid"] = ~np.any((Xa[:, up] < X0[:, up] - 1e-6), axis=1)
    res["syntactic_valid"] = res["onehot_valid"] & res["integer_valid"] & res["relation_valid"]
    res["threat_model_valid"] = res["syntactic_valid"] & res["immutable_untouched"] & res["direction_valid"]
    res["n_changed"] = changed.sum(1)
    return res


def blackbox(det, X, eps, prep, threat, queries=300, seed=0, thr=0.5):
    """Score-based l_inf random-search attack (Square-attack style) on vertices of
    the feasible box; works for any model exposing P(malicious)."""
    rng = np.random.default_rng(seed)
    X = np.asarray(X, np.float32)
    n, d = X.shape
    mut, up = masks(prep, threat, d)
    idx = np.where(mut)[0]
    if len(idx) == 0:
        return X.copy(), np.zeros(n, int)
    def vertex(shape_rows, cols):
        v = rng.choice([-1.0, 1.0], size=(shape_rows, len(cols))) * eps
        upc = up[cols]
        v[:, upc] = rng.choice([0.0, 1.0], size=(shape_rows, upc.sum())) * eps
        return v.astype(np.float32)
    def feasible(D):
        Xa = np.clip(X + D, 0, 1).astype(np.float32)
        return repair(prep, Xa, X) if threat == "constrained" else Xa
    D = np.zeros_like(X); D[:, idx] = vertex(n, idx)
    best_x = feasible(D); best = det.predict(best_x)
    used = np.ones(n, int)
    active = best >= thr
    for q in range(1, queries):
        if not active.any():
            break
        frac = max(1.0 / len(idx), 0.3 * (1 - q / queries))
        k = max(1, int(round(frac * len(idx))))
        rows = np.where(active)[0]
        Dn = D[rows].copy()
        m = len(rows)
        sel = np.argpartition(rng.random((m, len(idx))), k - 1, axis=1)[:, :k]
        cols = idx[sel]
        sign = rng.choice([-1.0, 1.0], size=(m, k))
        sign = np.where(up[cols], (sign > 0).astype(float), sign)
        Dn[np.arange(m)[:, None], cols] = (sign * eps).astype(np.float32)
        Xn = np.clip(X[rows] + Dn, 0, 1).astype(np.float32)
        if threat == "constrained":
            Xn = repair(prep, Xn, X[rows])
        s = det.predict(Xn)
        used[rows] += 1
        imp = s < best[rows]
        r_imp = rows[imp]
        D[r_imp] = Dn[imp]; best[r_imp] = s[imp]; best_x[r_imp] = Xn[imp]
        active = best >= thr
    return best_x, used


def attack_summary(det, X, y, Xa, thr, prep):
    p0, p1 = det.predict(X), det.predict(Xa)
    pred0, pred1 = (p0 >= thr).astype(int), (p1 >= thr).astype(int)
    elig = (y == 1) & (pred0 == 1)
    succ = elig & (pred1 == 0)
    v = validity(prep, Xa, X)
    linf = np.abs(Xa - X).max(1)
    return {"n_attack": int(len(y)), "eligible": int(elig.sum()), "success": int(succ.sum()),
            "asr": float(succ.sum() / max(elig.sum(), 1)),
            "mal_recall_clean": float(pred0[y == 1].mean()), "mal_recall_adv": float(pred1[y == 1].mean()),
            "mean_linf": float(linf[elig].mean()) if elig.any() else np.nan,
            "mean_features_changed": float(v["n_changed"][elig].mean()) if elig.any() else np.nan,
            "syntactic_valid_rate": float(v["syntactic_valid"][elig].mean()) if elig.any() else np.nan,
            "threat_model_valid_rate": float(v["threat_model_valid"][elig].mean()) if elig.any() else np.nan,
            "onehot_valid_rate": float(v["onehot_valid"][elig].mean()) if elig.any() else np.nan,
            "integer_valid_rate": float(v["integer_valid"][elig].mean()) if elig.any() else np.nan,
            "relation_valid_rate": float(v["relation_valid"][elig].mean()) if elig.any() else np.nan,
            "mean_p_drop": float((p0 - p1)[elig].mean()) if elig.any() else np.nan,
            "asr_syntactic_valid": float((succ & v["syntactic_valid"]).sum() / max(elig.sum(), 1)),
            "asr_threat_model_valid": float((succ & v["threat_model_valid"]).sum() / max(elig.sum(), 1))}, elig, succ

In [ ]:
%%writefile /kaggle/working/iotxai/explain.py
"""SHAP / LIME attributions, group-level aggregation, faithfulness under three
masking references with a random-ranking control, and stability metrics."""
from __future__ import annotations

import time

import numpy as np
import torch
import torch.nn as nn
from scipy.stats import rankdata


# ------------------------------------------------------------------ attributions
class _ProbWrap(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base
    def forward(self, x):
        return torch.sigmoid(self.base(x)).reshape(-1, 1)


def shap_values(det, X, background):
    """Signed attributions toward P(malicious), shape (n, d)."""
    import shap
    X = np.asarray(X, np.float32)
    if det.kind == "mlp":
        wrapped = _ProbWrap(det.obj).to(det.device).eval()
        bg = torch.as_tensor(np.asarray(background, np.float32), device=det.device)
        xx = torch.as_tensor(X, device=det.device)
        try:
            vals = shap.DeepExplainer(wrapped, bg).shap_values(xx, check_additivity=False)
        except Exception as exc:  # pragma: no cover
            print("DeepExplainer fallback to GradientExplainer:", exc)
            vals = shap.GradientExplainer(wrapped, bg).shap_values(xx)
    else:
        ex = shap.TreeExplainer(det.obj, data=np.asarray(background, np.float32),
                                model_output="probability", feature_perturbation="interventional")
        vals = ex.shap_values(X, check_additivity=False)
    if isinstance(vals, list):
        vals = vals[-1]
    vals = np.asarray(vals)
    if vals.ndim == 3:
        vals = vals[..., -1]
    return vals.astype(np.float32)


def lime_values(det, X, background, cat_cols, num_samples, seed):
    from lime.lime_tabular import LimeTabularExplainer
    X = np.asarray(X, np.float32)
    d = X.shape[1]
    ex = LimeTabularExplainer(np.asarray(background, np.float32), mode="classification",
                              categorical_features=list(cat_cols), discretize_continuous=False,
                              sample_around_instance=True, random_state=int(seed))
    def fn(a):
        p = det.predict(np.asarray(a, np.float32))
        return np.column_stack([1 - p, p])
    out = np.zeros((len(X), d), np.float32)
    for i in range(len(X)):
        e = ex.explain_instance(X[i], fn, labels=(1,), num_features=d, num_samples=num_samples)
        for j, w in e.local_exp[1]:
            out[i, j] = w
    return out


def to_groups(A, groups):
    return np.stack([A[:, g["cols"]].sum(1) for g in groups], axis=1).astype(np.float32)


# ------------------------------------------------------------------ faithfulness
class Masker:
    """Builds masked inputs for a set of group masks under three references."""

    def __init__(self, groups, d, bg, pool, n_marginal, n_knn, seed, device="cpu"):
        self.groups, self.d = groups, d
        self.G = len(groups)
        self.colmap = np.zeros((self.G, d), np.float32)
        for gi, g in enumerate(groups):
            self.colmap[gi, g["cols"]] = 1
        ref = np.median(bg, axis=0)
        for g in groups:      # categorical blocks: modal level pattern, not column medians
            if g["kind"] == "cat":
                B = bg[:, g["cols"]].round().astype(int)
                pats, cnt = np.unique(B, axis=0, return_counts=True)
                ref[g["cols"]] = pats[cnt.argmax()]
        self.median_ref = ref.astype(np.float32)
        self.bg = np.asarray(bg, np.float32)
        self.pool = np.asarray(pool, np.float32)
        self.n_marginal, self.n_knn = n_marginal, n_knn
        self.rng = np.random.default_rng(seed)
        self.device = device
        self.P = torch.as_tensor(self.pool, device=device)
        self.P2 = (self.P ** 2)

    def build(self, X, gmask, ref):
        """X (m,d), gmask (m,G) bool masked-groups -> (m*r, d) inputs, r replicates."""
        cm = (gmask.astype(np.float32) @ self.colmap) > 0  # (m,d)
        if ref == "median":
            return np.where(cm, self.median_ref[None, :], X).astype(np.float32), 1
        if ref == "marginal":
            r = self.n_marginal
            rows = self.rng.integers(0, len(self.bg), size=(len(X), r))
            R = self.bg[rows]                                   # (m,r,d)
            out = np.where(cm[:, None, :], R, X[:, None, :])
            return out.reshape(-1, self.d).astype(np.float32), r
        if ref == "knn":
            r = self.n_knn
            keep = torch.as_tensor((~cm).astype(np.float32), device=self.device)
            Xt = torch.as_tensor(X, device=self.device)
            res = []
            for i in range(0, len(X), 1024):
                k_, x_ = keep[i:i + 1024], Xt[i:i + 1024]
                dist = (k_ * x_ ** 2).sum(1, keepdim=True) - 2 * (k_ * x_) @ self.P.T + k_ @ self.P2.T
                nn_idx = torch.topk(dist, r, largest=False).indices.cpu().numpy()
                res.append(nn_idx)
            nn_idx = np.concatenate(res)
            R = self.pool[nn_idx]
            out = np.where(cm[:, None, :], R, X[:, None, :])
            return out.reshape(-1, self.d).astype(np.float32), r
        raise ValueError(ref)


def _topk_masks(order, k, G):
    m = np.zeros((len(order), G), bool)
    np.put_along_axis(m, order[:, :k], True, axis=1)
    return m


def faithfulness(det, X, evid, masker, k_fracs, refs, cls):
    """evid: (n,G) attribution in favour of the evaluated class ``cls`` (n,)
    (larger = more supporting).  Class score f_c = P(mal) if c==1 else 1-P(mal).
    Returns dict ref -> (comp_aopc[n], suff_aopc[n], comp[n,K], suff[n,K]);
    AOPC = mean over the k fractions.
    comprehensiveness_k = f_c(x) - E f_c(x with top-k groups replaced)   (higher = better)
    sufficiency_k      = f_c(x) - E f_c(x with all but top-k replaced) (lower = better)"""
    G = evid.shape[1]
    cls = np.asarray(cls).astype(int)
    def fc(p, c):
        return np.where(c == 1, p, 1.0 - p)
    base = fc(det.predict(X), cls)
    order = np.argsort(-evid, axis=1, kind="stable")
    out = {}
    for ref in refs:
        comp, suff = np.zeros((len(X), len(k_fracs))), np.zeros((len(X), len(k_fracs)))
        for ki, fr in enumerate(k_fracs):
            k = int(min(G - 1, max(1, round(fr * G))))
            top = _topk_masks(order, k, G)
            for which, M in (("comp", top), ("suff", ~top)):
                Xm, r = masker.build(X, M, ref)
                p = det.predict(Xm).reshape(len(X), r)
                p = fc(p, cls[:, None]).mean(1)
                (comp if which == "comp" else suff)[:, ki] = base - p
        out[ref] = (comp.mean(1), suff.mean(1), comp, suff)
    return out


# ------------------------------------------------------------------ stability
def _rank_rows(A):
    return np.apply_along_axis(rankdata, 1, A)


def stability(A, B, ks):
    """Row-wise comparisons of group attributions A,B (n,G), signed toward malicious."""
    absA, absB = np.abs(A), np.abs(B)
    res = {}
    oa, ob = np.argsort(-absA, 1, kind="stable"), np.argsort(-absB, 1, kind="stable")
    G = A.shape[1]
    for k in ks:
        k = min(k, G)
        ja = np.zeros(len(A))
        for i in range(len(A)):
            sa, sb = set(oa[i, :k]), set(ob[i, :k])
            ja[i] = len(sa & sb) / len(sa | sb)
        res[f"jaccard@{k}"] = ja
    ra, rb = _rank_rows(absA), _rank_rows(absB)
    ra -= ra.mean(1, keepdims=True); rb -= rb.mean(1, keepdims=True)
    den = np.sqrt((ra ** 2).sum(1) * (rb ** 2).sum(1))
    res["spearman"] = np.where(den > 0, (ra * rb).sum(1) / np.where(den > 0, den, 1), np.nan)
    na, nb = np.linalg.norm(A, axis=1), np.linalg.norm(B, axis=1)
    res["signed_cosine"] = np.where((na > 0) & (nb > 0), (A * B).sum(1) / np.maximum(na * nb, 1e-12), np.nan)
    res["top1_agree"] = (oa[:, 0] == ob[:, 0]).astype(float)
    return res


def timed(fn, *a, **k):
    t0 = time.perf_counter()
    r = fn(*a, **k)
    return r, time.perf_counter() - t0

In [ ]:
%%writefile /kaggle/working/iotxai/runner.py
"""Orchestration: one *run* = (dataset, split, seed).  Each run writes its own
folder under ``results_v2/runs`` and a ``done.json`` marker, so an interrupted
Kaggle session can be resumed and several sessions can be merged."""
from __future__ import annotations

import json
import zlib
import platform
import sys
import time
import traceback
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from . import attacks as A
from . import explain as E
from .config import Config
from .data import prepare
from .models import (Detector, binary_metrics, family_recall, fit_temperature, resolve_device,
                     set_seed, train_mlp, train_xgb, tune_threshold, ece_score)


class Logger:
    def __init__(self, path):
        self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
    def __call__(self, msg):
        line = f"[{time.strftime('%H:%M:%S')}] {msg}"
        print(line, flush=True)
        with open(self.path, "a") as f:
            f.write(line + "\n")


def environment():
    import sklearn, scipy, shap, xgboost, lime
    env = {"python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
           "cuda": torch.cuda.is_available(),
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
           "scipy": scipy.__version__, "shap": shap.__version__, "xgboost": xgboost.__version__,
           "lime": getattr(lime, "__version__", "unknown")}
    return env


# ---------------------------------------------------------------- model zoo
def build_models(z, cfg, seed, device, names, log):
    Xtr, ytr, Xva, yva = z["X_train"], z["y_train"], z["X_val"], z["y_val"]
    out, hist_rows, train_rows = {}, [], []
    for name in names:
        if name == "xgb":
            clf, secs = train_xgb(Xtr, ytr, Xva, yva, seed, cfg, device)
            det = Detector("xgb", clf, device); hist = []
        else:
            if name == "mlp_clean":
                kw = dict(mode="clean")
            elif name == "mlp_fgsmat":
                kw = dict(mode="at", eps=cfg.fgsmat_eps, lam=cfg.fgsmat_lambda, steps=1)
            elif name == "mlp_pgdat":
                kw = dict(mode="at", eps=cfg.pgdat_eps, lam=cfg.pgdat_lambda, steps=cfg.pgdat_steps)
            elif name.startswith("sweep_"):
                _, e, l = name.split("_")
                kw = dict(mode="at", eps=float(e), lam=float(l), steps=cfg.pgdat_steps)
            else:
                raise ValueError(name)
            net, hist, secs = train_mlp(Xtr, ytr, Xva, yva, seed, cfg, device, log=log, **kw)
            det = Detector("mlp", net, device)
            det.temperature = fit_temperature(det, Xva, yva)
        pv = det.predict(Xva)
        det.threshold, _ = tune_threshold(yva, pv)
        out[name] = det
        for h in hist:
            hist_rows.append(dict(model=name, **h))
        train_rows.append({"model": name, "train_seconds": secs, "val_tuned_threshold": det.threshold,
                           "temperature": det.temperature})
    return out, pd.DataFrame(hist_rows), pd.DataFrame(train_rows)


def predictive(dets, z, base):
    rows, fam_rows, probs = [], [], {}
    for name, det in dets.items():
        pv, pt = det.predict(z["X_val"]), det.predict(z["X_test"])
        probs[f"pval_{name}"] = pv.astype(np.float16); probs[f"ptest_{name}"] = pt.astype(np.float16)
        for tname, thr in (("fixed_0.5", 0.5), ("val_tuned", det.threshold)):
            m = binary_metrics(z["y_test"], pt, thr)
            rows.append(dict(base, model=name, threshold_rule=tname, **m))
            for fr in family_recall(z["y_test"], z["fam_test"], pt, thr):
                fam_rows.append(dict(base, model=name, threshold_rule=tname, **fr))
        if det.kind == "mlp":
            pc = det.predict_calibrated(z["X_test"])
            rows[-1]["ece_temperature_scaled"] = ece_score(z["y_test"], pc)
    return pd.DataFrame(rows), pd.DataFrame(fam_rows), probs


# ---------------------------------------------------------------- attacks
def run_attacks(dets, z, prep, cfg, base, seed, log, store_views):
    idx = z["attack_idx"]
    X, y, fam = z["X_test"][idx], z["y_test"][idx], z["fam_test"][idx]
    rows, fam_rows, advs = [], [], {}
    emax = max(cfg.epsilons)
    n_exp = len(z["explain_mal_idx"])
    for name, det in dets.items():
        for threat in cfg.threat_models:
            eps_list = list(cfg.epsilons) + (list(cfg.constrained_extra_eps) if threat == "constrained" else [])
            for eps in eps_list:
                kinds = ["bb"] if det.kind == "xgb" else ["fgsm", "pgd", "bb"]
                if eps > max(cfg.epsilons) and det.kind != "xgb":
                    kinds = ["pgd", "bb"]
                for kind in kinds:
                    t0 = time.perf_counter()
                    if kind == "bb":
                        Xa, q = A.blackbox(det, X, eps, prep, threat, cfg.bb_queries, seed=seed, thr=det.threshold)
                    else:
                        Xa = A.pgd(det, X, eps, prep, threat, steps=cfg.pgd_steps, seed=seed, fgsm=(kind == "fgsm"))
                        q = None
                    secs = time.perf_counter() - t0
                    s, elig, succ = A.attack_summary(det, X, y, Xa, det.threshold, prep)
                    s05, _, _ = A.attack_summary(det, X, y, Xa, 0.5, prep)
                    rows.append(dict(base, model=name, attack=kind, threat=threat, eps=eps, seconds=secs,
                                     mean_queries=float(q[elig].mean()) if q is not None and elig.any() else np.nan,
                                     asr_at_0p5=s05["asr"], **s))
                    for f in np.unique(fam):
                        m = fam == f
                        fam_rows.append(dict(base, model=name, attack=kind, threat=threat, eps=eps, family=f,
                                             eligible=int(elig[m].sum()), success=int(succ[m].sum())))
                    tag = f"{kind}_{threat[0]}_{eps:g}"
                    if store_views(name, tag):
                        advs[(name, tag)] = Xa[:n_exp]
                    if name == "mlp_clean" and kind == "pgd":   # transfer to the other detectors
                        for tgt, tdet in dets.items():
                            if tgt == name:
                                continue
                            st, _, _ = A.attack_summary(tdet, X, y, Xa, tdet.threshold, prep)
                            rows.append(dict(base, model=tgt, attack="transfer_pgd_from_mlp_clean", threat=threat,
                                             eps=eps, seconds=np.nan, mean_queries=np.nan, asr_at_0p5=np.nan, **st))
        log(f"  attacks done: {name}")
    return pd.DataFrame(rows), pd.DataFrame(fam_rows), advs


def view_plan(cfg, model_kind):
    emax = max(cfg.epsilons)
    cmax = max(cfg.constrained_extra_eps) if cfg.constrained_extra_eps else emax
    k = "bb" if model_kind == "xgb" else "pgd"
    if cfg.adv_views_reduced:
        return [f"{k}_c_{cmax:g}", f"{k}_n_{emax:g}"]
    if model_kind == "xgb":
        return [f"bb_c_{emax:g}", f"bb_c_{cmax:g}", f"bb_n_{emax:g}", f"bb_u_{emax:g}"]
    return [f"pgd_c_{e:g}" for e in cfg.epsilons] + [f"pgd_c_{cmax:g}", f"pgd_n_{emax:g}", f"pgd_u_{emax:g}"]


# ---------------------------------------------------------------- explanations
def run_explanations(dets, advs, z, prep, cfg, base, seed, device, log, arrays):
    groups = prep.groups
    cat_cols = [j for j, k in enumerate(prep.col_kind) if k == "cat"]
    Xtr = z["X_train"]
    bg = Xtr[z["shap_bg_idx"]]; lime_bg = Xtr[z["lime_bg_idx"]]; pool = Xtr[z["knn_pool_idx"]]
    ben, mal = z["explain_benign_idx"], z["explain_mal_idx"]
    Xc = np.concatenate([z["X_test"][ben], z["X_test"][mal]])
    yc = np.concatenate([z["y_test"][ben], z["y_test"][mal]])
    fc = np.concatenate([z["fam_test"][ben], z["fam_test"][mal]])
    n_b = len(ben)
    refs = ["median", "marginal", "knn"]
    rec_rows, rt_rows = [], []
    dev_knn = str(device) if device.type == "cuda" else "cpu"
    for mname, det in dets.items():
        views = {"clean": Xc}
        for tag in view_plan(cfg, det.kind):
            views[tag] = advs[(mname, tag)]
        clean_groups = {}
        for vname, Xv in views.items():
            masker = E.Masker(groups, Xv.shape[1], bg, pool, cfg.marginal_draws, cfg.knn_neighbors,
                              seed=seed * 7 + len(vname), device=dev_knn)
            p = det.predict(Xv); cls = (p >= det.threshold).astype(int)
            yv = yc if vname == "clean" else np.ones(len(Xv), int)
            fv = fc if vname == "clean" else fc[n_b:]
            # random-ranking control (shared by both explainers)
            rng = np.random.default_rng(seed + 991)
            rand = {r: [np.zeros(len(Xv)), np.zeros(len(Xv))] for r in refs}
            for _ in range(cfg.random_order_draws):
                ev = rng.random((len(Xv), len(groups)))
                fr = E.faithfulness(det, Xv, ev, masker, cfg.k_fracs, refs, cls)
                for r in refs:
                    rand[r][0] += fr[r][0] / cfg.random_order_draws
                    rand[r][1] += fr[r][1] / cfg.random_order_draws
            for ex in ["SHAP", "LIME"]:
                if ex == "SHAP":
                    Araw, secs = E.timed(E.shap_values, det, Xv, bg)
                else:
                    Araw, secs = E.timed(E.lime_values, det, Xv, lime_bg, cat_cols, cfg.lime_num_samples,
                                         seed=seed * 1000 + zlib.crc32(vname.encode()) % 997)
                Ag = E.to_groups(Araw, groups)
                arrays[f"attr__{mname}__{ex}__{vname}"] = Ag.astype(np.float16)
                rt_rows.append(dict(base, model=mname, explainer=ex, view=vname, n=len(Xv), seconds=secs,
                                    ms_per_record=1000 * secs / len(Xv), device=str(device)))
                evid = np.where(cls[:, None] == 1, Ag, -Ag)
                fa = E.faithfulness(det, Xv, evid, masker, cfg.k_fracs, refs, cls)
                # v1-style variant: malicious score, |attribution| ranking, median reference
                v1 = E.faithfulness(det, Xv, np.abs(Ag), masker, cfg.k_fracs, ["median"], np.ones(len(Xv), int))
                rec = pd.DataFrame(dict(rec=np.arange(len(Xv)), y=yv, family=fv, p_mal=p, cls=cls))
                for r in refs:
                    rec[f"comp_{r}"], rec[f"suff_{r}"] = fa[r][0], fa[r][1]
                    rec[f"comp_rand_{r}"], rec[f"suff_rand_{r}"] = rand[r][0], rand[r][1]
                rec["comp_v1style"] = v1["median"][0]
                if vname == "clean":
                    clean_groups[ex] = Ag
                else:
                    Ac = clean_groups[ex][n_b:]
                    st = E.stability(Ac, Ag, cfg.stability_ks + [max(1, int(round(0.2 * len(groups))))])
                    for k_, v_ in st.items():
                        rec[f"stab_{k_}"] = v_
                    p0 = det.predict(Xc[n_b:])
                    rec["evaded"] = ((p0 >= det.threshold) & (p < det.threshold)).astype(int)
                for k_, v_ in base.items():
                    rec[k_] = v_
                rec["model"], rec["explainer"], rec["view"] = mname, ex, vname
                rec_rows.append(rec)
        log(f"  explanations done: {mname}")
    return pd.concat(rec_rows, ignore_index=True), pd.DataFrame(rt_rows)


def run_repeats(dets, z, prep, cfg, base, seed, log):
    """Repeated-call LIME (different sampling seeds) and SHAP background resampling."""
    cat_cols = [j for j, k in enumerate(prep.col_kind) if k == "cat"]
    X = z["X_test"][z["explain_mal_idx"][:cfg.repeat_rows]]
    Xtr = z["X_train"]
    rows = []
    for mname, det in dets.items():
        for ex in ["LIME", "SHAP"]:
            reps = []
            R = cfg.lime_repeats if ex == "LIME" else cfg.shap_bg_repeats
            for r in range(R):
                if ex == "LIME":
                    A_ = E.lime_values(det, X, Xtr[z["lime_bg_idx"]], cat_cols, cfg.lime_num_samples,
                                       seed=seed * 7919 + r)
                else:
                    rng = np.random.default_rng(seed * 104729 + r)
                    bg = Xtr[rng.choice(len(Xtr), cfg.shap_background_rows, replace=False)]
                    A_ = E.shap_values(det, X, bg)
                reps.append(E.to_groups(A_, prep.groups))
            for a, b in combinations(range(R), 2):
                st = E.stability(reps[a], reps[b], cfg.stability_ks)
                rows.append(dict(base, model=mname, explainer=ex, source="lime_sampling" if ex == "LIME" else "shap_background",
                                 rep_a=a, rep_b=b, **{k: float(np.nanmean(v)) for k, v in st.items()}))
        log(f"  repeats done: {mname}")
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- one run
def run_one(cfg: Config, dataset, split_id, seed, log, explain=True, repeats=False):
    out = Path(cfg.out_dir); work = Path(cfg.work_dir)
    rdir = out / "runs" / f"{dataset}_split{split_id}_seed{seed}"
    if (rdir / "done.json").exists():
        log(f"skip {rdir.name} (done)"); return
    rdir.mkdir(parents=True, exist_ok=True)
    device = resolve_device(cfg.device)
    t_run = time.time()
    z, prep = prepare(cfg, dataset, split_id, work, out, log)
    base = {"dataset": dataset, "split": split_id, "seed": seed}
    log(f"RUN {rdir.name} on {device}: d={z['X_train'].shape[1]} groups={len(prep.groups)}")
    dets, hist, trainrows = build_models(z, cfg, seed, device, cfg.models, log)
    for k, v in base.items():
        hist[k] = v; trainrows[k] = v
    hist.to_csv(rdir / "training_history.csv", index=False)
    trainrows.to_csv(rdir / "training.csv", index=False)
    pm, fm, probs = predictive(dets, z, base)
    pm.to_csv(rdir / "predictive.csv", index=False); fm.to_csv(rdir / "predictive_family.csv", index=False)
    arrays = dict(probs)
    wanted = set()
    for name, det in dets.items():
        for v in view_plan(cfg, det.kind):
            wanted.add((name, v))
    am, af, advs = run_attacks(dets, z, prep, cfg, base, seed, log,
                               store_views=lambda n, t: explain and (n, t) in wanted)
    am.to_csv(rdir / "attacks.csv", index=False); af.to_csv(rdir / "attacks_family.csv", index=False)
    for (n, t), Xa in advs.items():
        arrays[f"adv__{n}__{t}"] = Xa.astype(np.float16)
    if explain:
        rec, rt = run_explanations(dets, advs, z, prep, cfg, base, seed, device, log, arrays)
        rec.to_csv(rdir / "explanations_records.csv.gz", index=False)
        from .analysis import seed_level_explanations
        seed_level_explanations(rec).to_csv(rdir / "explanations_seed.csv", index=False)
        rt.to_csv(rdir / "explanations_runtime.csv", index=False)
    if repeats:
        run_repeats(dets, z, prep, cfg, base, seed, log).to_csv(rdir / "repeats.csv", index=False)
    np.savez_compressed(rdir / "arrays.npz", **arrays)
    mdir = rdir / "models"; mdir.mkdir(exist_ok=True)
    for name, det in dets.items():
        if det.kind == "mlp":
            torch.save(det.obj.state_dict(), mdir / f"{name}.pt")
        else:
            det.obj.save_model(str(mdir / f"{name}.json"))
    (rdir / "done.json").write_text(json.dumps({"seconds": time.time() - t_run, **base}))
    log(f"DONE {rdir.name} in {time.time() - t_run:.0f}s")
    if cfg.print_tables:
        from .report import print_run
        print_run(rdir)


def run_all(cfg: Config, log=None):
    out = Path(cfg.out_dir); out.mkdir(parents=True, exist_ok=True)
    Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
    log = log or Logger(out / "log.txt")
    cfg.to_json(out / f"config_{int(time.time())}.json")
    (out / "environment.json").write_text(json.dumps(environment(), indent=2))
    for ds in cfg.datasets:
        for sp in cfg.split_ids:
            for i, seed in enumerate(cfg.seeds):
                try:
                    run_one(cfg, ds, sp, seed, log, explain=cfg.explain, repeats=(i < cfg.repeat_seeds and sp == 0))
                except Exception:
                    log("ERROR in run " + f"{ds}/{sp}/{seed}\n" + traceback.format_exc())
                    raise

In [ ]:
%%writefile /kaggle/working/iotxai/extras.py
"""Supplementary experiments requested in review:
(1) adversarial-training sensitivity sweep (robustness-accuracy trade-off),
(2) multiclass attack-family detection with targeted/untargeted evasion and
    attribution stability, (3) explanation/inference resource profiling."""
from __future__ import annotations

import json
import os
import threading
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

from . import attacks as A
from . import explain as E
from .data import prepare
from .models import Detector, MLP, binary_metrics, resolve_device, set_seed, train_mlp, train_xgb
from .runner import Logger, build_models
from .report import print_block


# ------------------------------------------------------------ (1) AT sweep
def at_sweep(cfg, log):
    out = Path(cfg.out_dir) / "extras"; out.mkdir(parents=True, exist_ok=True)
    device = resolve_device(cfg.device)
    rows = []
    names = ["mlp_clean"] + [f"sweep_{e:g}_{l:g}" for e in cfg.at_sweep_eps for l in cfg.at_sweep_lambda]
    for ds in cfg.datasets:
        z, prep = prepare(cfg, ds, 0, Path(cfg.work_dir), Path(cfg.out_dir), log)
        idx = z["attack_idx"]; X, y = z["X_test"][idx], z["y_test"][idx]
        for seed in cfg.seeds[:cfg.at_sweep_seeds]:
            dets, _, tr = build_models(z, cfg, seed, device, names, log)
            for name, det in dets.items():
                pt = det.predict(z["X_test"])
                m = binary_metrics(z["y_test"], pt, det.threshold)
                m05 = binary_metrics(z["y_test"], pt, 0.5)
                for threat in cfg.threat_models:
                    for eps in cfg.epsilons:
                        Xa = A.pgd(det, X, eps, prep, threat, steps=cfg.pgd_steps, seed=seed)
                        s, _, _ = A.attack_summary(det, X, y, Xa, det.threshold, prep)
                        rows.append({"dataset": ds, "seed": seed, "model": name, "threat": threat, "eps": eps,
                                     "clean_macro_f1": m["macro_f1"], "clean_macro_f1_at_0p5": m05["macro_f1"],
                                     "clean_fpr": m["fpr"], "clean_fnr": m["fnr"], "threshold": det.threshold,
                                     "asr": s["asr"], "mal_recall_adv": s["mal_recall_adv"]})
            log(f"sweep {ds} seed {seed} done")
            pd.DataFrame(rows).to_csv(out / "at_sweep.csv", index=False)
    print_block("extras", "at_sweep", pd.DataFrame(rows))
    return pd.DataFrame(rows)


# ------------------------------------------------------------ (2) multiclass
class _SoftWrap(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base
    def forward(self, x):
        return torch.softmax(self.base(x), dim=-1)


def _mc_pgd(model, X, eps, prep, threat, steps, target=None, true=None, device="cpu", seed=0):
    mut_np, up_np = A.masks(prep, threat, X.shape[1])
    mut = torch.as_tensor(mut_np, device=device, dtype=torch.float32)
    up = torch.as_tensor(up_np, device=device)
    x0 = torch.as_tensor(X, dtype=torch.float32, device=device)
    g = torch.Generator().manual_seed(seed)
    xa = A._project(x0 + (torch.rand(x0.shape, generator=g).to(device) * 2 - 1) * eps, x0, eps, mut, up)
    alpha = 2.5 * eps / steps
    ce = nn.CrossEntropyLoss(reduction="sum")
    for _ in range(steps):
        xa.requires_grad_(True)
        logits = model(xa)
        if target is not None:   # targeted: decrease loss toward target
            loss = -ce(logits, torch.full((len(X),), target, device=device, dtype=torch.long))
        else:                    # untargeted: increase loss of true class
            loss = ce(logits, torch.as_tensor(true, device=device, dtype=torch.long))
        grad = torch.autograd.grad(loss, xa)[0]
        xa = A._project(xa.detach() + alpha * grad.sign(), x0, eps, mut, up)
    Xa = xa.detach().cpu().numpy()
    if threat == "constrained":
        Xa = A.repair(prep, Xa, X)
    return Xa


def multiclass(cfg, log):
    import shap
    out = Path(cfg.out_dir) / "extras"; out.mkdir(parents=True, exist_ok=True)
    device = resolve_device(cfg.device)
    met_rows, att_rows, stab_rows, cls_rows = [], [], [], []
    emax = max(cfg.epsilons)
    for ds in cfg.datasets:
        z, prep = prepare(cfg, ds, 0, Path(cfg.work_dir), Path(cfg.out_dir), log)
        classes = sorted(np.unique(z["fam_train"]))
        cmap = {c: i for i, c in enumerate(classes)}
        benign_cls = cmap.get("normal", None)
        yt = lambda f: np.array([cmap.get(v, -1) for v in f])
        ytr, yva, yte = yt(z["fam_train"]), yt(z["fam_val"]), yt(z["fam_test"])
        mal = z["explain_mal_idx"]
        X, ytrue = z["X_test"][mal], yte[mal]
        for seed in cfg.seeds[:cfg.multiclass_seeds]:
            net, hist, secs = train_mlp(z["X_train"], ytr, z["X_val"], yva, seed, cfg, device,
                                        n_classes=len(classes), log=log)
            det = Detector("mlp", net, device)
            pred = det.logits(z["X_test"]).argmax(1)
            met_rows.append({"dataset": ds, "seed": seed, "model": "mlp_multiclass",
                             "macro_f1": f1_score(yte, pred, average="macro"),
                             "weighted_f1": f1_score(yte, pred, average="weighted"),
                             "binary_collapse_macro_f1": f1_score(yte != benign_cls, pred != benign_cls, average="macro")})
            per = f1_score(yte, pred, average=None, labels=list(range(len(classes))))
            for c, f in zip(classes, per):
                cls_rows.append({"dataset": ds, "seed": seed, "model": "mlp_multiclass", "family": c, "f1": f})
            clf, _ = train_xgb(z["X_train"], ytr, z["X_val"], yva, seed, cfg, device, n_classes=len(classes))
            px = clf.predict_proba(z["X_test"]).argmax(1)
            met_rows.append({"dataset": ds, "seed": seed, "model": "xgb_multiclass",
                             "macro_f1": f1_score(yte, px, average="macro"),
                             "weighted_f1": f1_score(yte, px, average="weighted"),
                             "binary_collapse_macro_f1": f1_score(yte != benign_cls, px != benign_cls, average="macro")})
            # attacks on the 300 explained malicious records
            p0 = det.logits(X).argmax(1)
            correct = p0 == ytrue
            wrapped = _SoftWrap(net).to(device).eval()
            bg = torch.as_tensor(z["X_train"][z["shap_bg_idx"]], device=device)
            expl = shap.DeepExplainer(wrapped, bg)
            def attr(Xin):
                v = np.asarray(expl.shap_values(torch.as_tensor(Xin, device=device), check_additivity=False))
                if v.ndim == 3 and v.shape[0] == len(classes) and v.shape[1] == len(Xin):
                    v = np.transpose(v, (1, 2, 0))
                return v  # (n, d, C)
            Ac = attr(X)
            Ac_true = E.to_groups(Ac[np.arange(len(X)), :, ytrue], prep.groups)
            for threat in cfg.threat_models:
                for mode in ["untargeted", "to_benign"]:
                    if mode == "to_benign" and benign_cls is None:
                        continue
                    Xa = _mc_pgd(net, X, emax, prep, threat, cfg.pgd_steps,
                                 target=benign_cls if mode == "to_benign" else None, true=ytrue,
                                 device=device, seed=seed)
                    p1 = det.logits(Xa).argmax(1)
                    succ = correct & ((p1 != ytrue) if mode == "untargeted" else (p1 == benign_cls))
                    fams = np.array(classes)[ytrue]
                    for f in np.unique(fams):
                        m = fams == f
                        att_rows.append({"dataset": ds, "seed": seed, "threat": threat, "mode": mode, "eps": emax,
                                         "family": f, "eligible": int(correct[m].sum()), "success": int(succ[m].sum())})
                    Aa = attr(Xa)
                    Aa_true = E.to_groups(Aa[np.arange(len(X)), :, ytrue], prep.groups)
                    st = E.stability(Ac_true, Aa_true, cfg.stability_ks)
                    for k, v in st.items():
                        stab_rows.append({"dataset": ds, "seed": seed, "threat": threat, "mode": mode,
                                          "metric": k, "mean": float(np.nanmean(v)),
                                          "mean_evaded": float(np.nanmean(v[succ])) if succ.any() else np.nan})
            log(f"multiclass {ds} seed {seed} done")
            pd.DataFrame(met_rows).to_csv(out / "multiclass_metrics.csv", index=False)
            pd.DataFrame(cls_rows).to_csv(out / "multiclass_per_class_f1.csv", index=False)
            pd.DataFrame(att_rows).to_csv(out / "multiclass_attacks.csv", index=False)
            pd.DataFrame(stab_rows).to_csv(out / "multiclass_stability.csv", index=False)
    for nm, rr in (("multiclass_metrics", met_rows), ("multiclass_per_class_f1", cls_rows),
                   ("multiclass_attacks", att_rows), ("multiclass_stability", stab_rows)):
        print_block("extras", nm, pd.DataFrame(rr))


# ------------------------------------------------------------ (3) profiling
class PeakRSS:
    def __init__(self, interval=0.005):
        import psutil
        self.p = psutil.Process(os.getpid()); self.interval = interval
    def __enter__(self):
        self.start = self.p.memory_info().rss; self.peak = self.start; self._run = True
        self.t = threading.Thread(target=self._loop, daemon=True); self.t.start(); return self
    def _loop(self):
        while self._run:
            self.peak = max(self.peak, self.p.memory_info().rss); time.sleep(self.interval)
    def __exit__(self, *a):
        self._run = False; self.t.join()
        self.peak = max(self.peak, self.p.memory_info().rss)
        self.delta_mb = (self.peak - self.start) / 2 ** 20
        self.peak_mb = self.peak / 2 ** 20
        self.start_mb = self.start / 2 ** 20


def profiling(cfg, log):
    out = Path(cfg.out_dir) / "extras"; out.mkdir(parents=True, exist_ok=True)
    rows = []
    gpu = torch.cuda.is_available()
    for ds in cfg.datasets:
        z, prep = prepare(cfg, ds, 0, Path(cfg.work_dir), Path(cfg.out_dir), log)
        seed = cfg.seeds[0]
        dev0 = resolve_device(cfg.device)
        dets, _, _ = build_models(z, cfg, seed, dev0, ["mlp_clean", "xgb"], log)
        X = z["X_test"][z["explain_mal_idx"][:cfg.profile_rows]]
        bg = z["X_train"][z["shap_bg_idx"]]; lbg = z["X_train"][z["lime_bg_idx"]]
        cat_cols = [j for j, k in enumerate(prep.col_kind) if k == "cat"]
        settings = [("cpu", 1), ("cpu", os.cpu_count() or 1)] + ([("cuda", None)] if gpu else [])
        for devname, threads in settings:
            if threads:
                torch.set_num_threads(threads)
            dev = torch.device(devname)
            for mname, det in dets.items():
                if det.kind == "mlp":
                    det.obj.to(dev); det.device = dev
                elif devname == "cuda":
                    continue  # tree models profiled on CPU only
                else:
                    det.obj.set_params(n_jobs=threads)
                # inference latency
                x1 = X[:1]
                for _ in range(20):
                    det.predict(x1)
                t0 = time.perf_counter()
                for _ in range(500):
                    det.predict(x1)
                lat1 = (time.perf_counter() - t0) / 500
                Xb = np.repeat(X, 11, axis=0)[:1024]
                t0 = time.perf_counter(); det.predict(Xb); latb = time.perf_counter() - t0
                rows.append({"dataset": ds, "device": devname, "threads": threads, "model": mname, "task": "inference",
                             "ms_per_record": 1000 * lat1, "batch1024_ms_per_record": 1000 * latb / 1024})
                tasks = [("SHAP", lambda: E.shap_values(det, X, bg)),
                         ("LIME", lambda: E.lime_values(det, X, lbg, cat_cols, cfg.lime_num_samples, seed)),
                         ("BB_attack_c", lambda: A.blackbox(det, X, max(cfg.epsilons), prep, "constrained",
                                                           cfg.bb_queries, seed, det.threshold))]
                if det.kind == "mlp":
                    tasks.append(("PGD_attack_c", lambda: A.pgd(det, X, max(cfg.epsilons), prep, "constrained",
                                                               cfg.pgd_steps, seed)))
                if det.kind == "mlp":
                    model_kb = sum(p.numel() * p.element_size() for p in det.obj.parameters()) / 1024
                else:
                    model_kb = len(det.obj.get_booster().save_raw("json")) / 1024
                for tname, fn in tasks:
                    import gc, tracemalloc
                    gc.collect()
                    if gpu:
                        torch.cuda.reset_peak_memory_stats()
                    tracemalloc.start()
                    with PeakRSS() as pr:
                        t0 = time.perf_counter(); fn(); secs = time.perf_counter() - t0
                    tm_peak = tracemalloc.get_traced_memory()[1] / 2 ** 20
                    tracemalloc.stop()
                    rows.append({"dataset": ds, "device": devname, "threads": threads, "model": mname, "task": tname,
                                 "n": len(X), "seconds": secs, "ms_per_record": 1000 * secs / len(X),
                                 "peak_rss_delta_mb": pr.delta_mb, "process_rss_before_mb": pr.start_mb,
                                 "process_rss_peak_mb": pr.peak_mb, "py_alloc_peak_mb": tm_peak,
                                 "model_size_kb": model_kb,
                                 "gpu_peak_alloc_mb": torch.cuda.max_memory_allocated() / 2 ** 20 if (gpu and devname == "cuda") else np.nan})
                    log(f"profile {ds} {devname}/{threads} {mname} {tname}: {1000*secs/len(X):.1f} ms/rec")
                    pd.DataFrame(rows).to_csv(out / "profiling.csv", index=False)
        torch.set_num_threads(os.cpu_count() or 1)
    print_block("extras", "profiling", pd.DataFrame(rows))
    return pd.DataFrame(rows)


# ------------------------------------------------------------ (4) baseline ablation
def baseline_ablation(cfg, log, combos=None, out_name="baseline_ablation"):
    """Isolates the contributors to the v1 Edge-IIoTset baseline deficit: feature
    policy (numeric-only vs. numeric+categorical), de-duplication, and threshold rule."""
    import dataclasses
    out = Path(cfg.out_dir) / "extras"; out.mkdir(parents=True, exist_ok=True)
    device = resolve_device(cfg.device)
    rows = []
    combos = combos or [(p, d, "logminmax") for p in ["numeric_only", "v2"] for d in [False, True]]
    for policy, dedup, scaling in combos:
        if True:
            c2 = dataclasses.replace(cfg, feature_policy=policy, dedup=dedup, scaling=scaling)
            for ds in cfg.datasets:
                z, prep = prepare(c2, ds, 0, Path(cfg.work_dir), Path(cfg.out_dir), log)
                for seed in cfg.seeds[:cfg.at_sweep_seeds]:
                    dets, _, _ = build_models(z, c2, seed, device, ["mlp_clean", "xgb"], log)
                    for name, det in dets.items():
                        pt = det.predict(z["X_test"])
                        for rule, thr in (("fixed_0.5", 0.5), ("val_tuned", det.threshold)):
                            m = binary_metrics(z["y_test"], pt, thr)
                            rows.append({"dataset": ds, "feature_policy": policy, "dedup": dedup, "scaling": scaling, "seed": seed,
                                         "model": name, "threshold_rule": rule, "n_columns": z["X_train"].shape[1],
                                         "n_train": len(z["y_train"]), **m})
                    log(f"ablation {ds} {policy} dedup={dedup} {scaling} seed {seed} done")
                    pd.DataFrame(rows).to_csv(out / f"{out_name}.csv", index=False)
    print_block("extras", out_name, pd.DataFrame(rows))
    return pd.DataFrame(rows)


# ------------------------------------------------------------ (5) operating point
def operating_point(cfg, log):
    """Attack success as a function of the decision threshold. Thresholds are matched
    on the validation benign records to a target false-positive rate, so detectors are
    compared at equal false-alarm cost.  PGD minimises the logit independently of the
    threshold, so one adversarial set is scored at every threshold; the black-box attack
    early-stops at the threshold and is re-run per threshold."""
    out = Path(cfg.out_dir) / "extras"; out.mkdir(parents=True, exist_ok=True)
    device = resolve_device(cfg.device)
    rows = []
    targets = [0.001, 0.005, 0.01, 0.02, 0.05]
    settings = [("unconstrained", max(cfg.epsilons)), ("numeric", max(cfg.epsilons)),
                ("constrained", max(cfg.epsilons)), ("constrained", 1.0)]
    for ds in cfg.datasets:
        z, prep = prepare(cfg, ds, 0, Path(cfg.work_dir), Path(cfg.out_dir), log)
        idx = z["attack_idx"]; X, y = z["X_test"][idx], z["y_test"][idx]
        ben_val = z["X_val"][z["y_val"] == 0]
        for seed in cfg.seeds[:cfg.at_sweep_seeds]:
            dets, _, _ = build_models(z, cfg, seed, device, ["mlp_clean", "mlp_fgsmat", "mlp_pgdat", "xgb"], log)
            for name, det in dets.items():
                pv_ben = det.predict(ben_val)
                pt = det.predict(z["X_test"])
                thr = {"val_tuned": det.threshold, "fixed_0.5": 0.5}
                for t in targets:
                    thr[f"fpr_{t:g}"] = float(np.quantile(pv_ben, 1 - t))
                for threat, eps in settings:
                    if det.kind == "mlp":
                        Xa = A.pgd(det, X, eps, prep, threat, steps=cfg.pgd_steps, seed=seed)
                    for tname, tv in thr.items():
                        if det.kind != "mlp":
                            Xa, _ = A.blackbox(det, X, eps, prep, threat, cfg.bb_queries, seed=seed, thr=tv)
                        s, _, _ = A.attack_summary(det, X, y, Xa, tv, prep)
                        m = binary_metrics(z["y_test"], pt, tv)
                        rows.append({"dataset": ds, "seed": seed, "model": name, "threat": threat, "eps": eps,
                                     "threshold_rule": tname, "threshold": tv, "test_fpr": m["fpr"],
                                     "test_mal_recall": m["mal_recall"], "test_macro_f1": m["macro_f1"],
                                     "eligible": s["eligible"], "asr": s["asr"], "mal_recall_adv": s["mal_recall_adv"]})
            log(f"operating point {ds} seed {seed} done")
            pd.DataFrame(rows).to_csv(out / "operating_point.csv", index=False)
    print_block("extras", "operating_point", pd.DataFrame(rows))
    return pd.DataFrame(rows)


def v1_path(cfg, log):
    """Reproduces the original (v1) pipeline and changes one factor at a time:
    P0 v1 features + v1 quantile scaling, no de-duplication (= original submission)
    P1 v1 features + log/min-max scaling
    (P2-P4, i.e. v2 numeric-only / +categorical / +de-duplication, are in baseline_ablation)."""
    return baseline_ablation(cfg, log, combos=[("v1", False, "quantile_v1"), ("v1", False, "logminmax")],
                             out_name="baseline_v1_path")

In [ ]:
%%writefile /kaggle/working/iotxai/analysis.py
"""Seed-level statistics, tables and figures for the revised manuscript.

Usage:  python -m iotxai.analysis <results_dir> [<results_dir> ...] --out <dir>

Inference unit = training seed (one aggregated value per seed and cell).
Primary outcomes (pre-declared in the revision):
  P1 attack success rate (ASR), PGD for MLPs / black-box for XGBoost, both threat models;
  P2 faithfulness = comprehensiveness AOPC minus the random-ranking control,
     marginal-sampling reference, evaluated class = predicted class;
  P3 stability = Spearman rank correlation of |group attribution| between the
     clean record and its attacked version (constrained threat model, eps = max).
Tests: two-sided exact Wilcoxon signed-rank on seed-paired differences, Holm
correction within each predeclared family, Cohen's d_z, percentile bootstrap CI
over seeds (10 000 resamples).
"""
from __future__ import annotations

import argparse
import json
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

MODEL_LABEL = {"mlp_clean": "MLP", "mlp_fgsmat": "MLP+FGSM-AT", "mlp_pgdat": "MLP+PGD-AT", "xgb": "XGBoost"}
DS_LABEL = {"ton": "ToN-IoT", "edge": "Edge-IIoTset"}
EPS_MAIN = 0.05          # primary l_inf budget (pre-declared)
THREAT_ABBR = {"unconstrained": "U", "numeric": "N", "constrained": "C"}


# ------------------------------------------------------------------ loading
def load(dirs):
    frames = {}
    names = ["predictive", "predictive_family", "attacks", "attacks_family", "training",
             "explanations_runtime", "repeats", "training_history"]
    for nm in names + ["explanations_records", "explanations_seed"]:
        parts = []
        for d in dirs:
            for f in sorted(Path(d).glob(f"runs/*/{nm}.csv*")):
                parts.append(pd.read_csv(f))
        frames[nm] = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    extras = {}
    for d in dirs:
        for f in Path(d).glob("extras/*.csv"):
            extras.setdefault(f.stem, []).append(pd.read_csv(f))
    frames["extras"] = {k: pd.concat(v, ignore_index=True) for k, v in extras.items()}
    audits = {}
    for d in dirs:
        for f in Path(d).glob("data_audit/*.json"):
            audits[f.stem] = json.loads(f.read_text())
    frames["audit"] = audits
    return frames


# ------------------------------------------------------------------ stats helpers
def boot_ci(x, n=10000, seed=0):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    m = rng.choice(x, size=(n, len(x)), replace=True).mean(1)
    return tuple(np.percentile(m, [2.5, 97.5]))


def paired(diff):
    d = np.asarray(diff, float); d = d[np.isfinite(d)]
    out = {"n": len(d), "mean_diff": d.mean() if len(d) else np.nan,
           "sd_diff": d.std(ddof=1) if len(d) > 1 else np.nan}
    out["dz"] = out["mean_diff"] / out["sd_diff"] if out["sd_diff"] and out["sd_diff"] > 0 else np.nan
    out["ci_low"], out["ci_high"] = boot_ci(d)
    out["n_positive"] = int((d > 0).sum())
    if len(d) >= 2 and np.any(d != 0):
        try:
            out["p_wilcoxon"] = stats.wilcoxon(d, method="exact" if len(d) <= 25 else "auto").pvalue
        except Exception:
            out["p_wilcoxon"] = stats.wilcoxon(d).pvalue
        out["p_ttest"] = stats.ttest_1samp(d, 0).pvalue
    else:
        out["p_wilcoxon"] = out["p_ttest"] = np.nan
    return out


def holm(p):
    p = np.asarray(p, float); m = np.isfinite(p).sum()
    order = np.argsort(np.where(np.isfinite(p), p, np.inf))
    adj = np.full_like(p, np.nan); running = 0
    for rank, i in enumerate(order):
        if not np.isfinite(p[i]):
            continue
        running = max(running, min(1, (m - rank) * p[i]))
        adj[i] = running
    return adj


def mde_dz(n, alpha=0.05, power=0.8):
    """Minimum detectable paired effect (d_z) for a two-sided one-sample t-test."""
    from statsmodels.stats.power import TTestPower
    return TTestPower().solve_power(nobs=n, alpha=alpha, power=power, alternative="two-sided")


def msd(x, digits=4):
    x = np.asarray(x, float)
    return f"{np.nanmean(x):.{digits}f} $\\pm$ {np.nanstd(x, ddof=1):.{digits}f}"


# ------------------------------------------------------------------ seed-level tables
def seed_level_explanations(rec):
    rec = rec.copy()
    for r in ["median", "marginal", "knn"]:
        rec[f"dcomp_{r}"] = rec[f"comp_{r}"] - rec[f"comp_rand_{r}"]
        rec[f"dsuff_{r}"] = rec[f"suff_{r}"] - rec[f"suff_rand_{r}"]
    rec["neg_v1"] = (rec["comp_v1style"] < 0).astype(float)
    rec["neg_pred_median"] = (rec["comp_median"] < 0).astype(float)
    rec["neg_pred_marginal"] = (rec["comp_marginal"] < 0).astype(float)
    keys = ["dataset", "split", "seed", "model", "explainer", "view"]
    vals = [c for c in rec.columns if c.startswith(("comp", "suff", "dcomp", "dsuff", "stab_", "neg_"))]
    g = rec.groupby(keys)[vals].mean().reset_index()
    ev = rec[rec["view"] != "clean"].copy()
    if len(ev):
        sub = ev[ev["evaded"] == 1].groupby(keys)[[c for c in vals if c.startswith("stab_")]].mean()
        sub.columns = [c + "_evaded" for c in sub.columns]
        g = g.merge(sub.reset_index(), on=keys, how="left")
        g = g.merge(ev.groupby(keys)["evaded"].mean().rename("evaded_rate").reset_index(), on=keys, how="left")
    return g


def cross_seed_stability(dirs, ks=(3, 5)):
    from .explain import stability
    rows = []
    for d in dirs:
        runs = sorted(Path(d).glob("runs/*/arrays.npz"))
        by = {}
        for f in runs:
            ds, sp, sd = f.parent.name.split("_")
            by.setdefault((ds, sp), []).append((int(sd.replace("seed", "")), f))
        for (ds, sp), lst in by.items():
            arrs = {s: np.load(f) for s, f in lst}
            keys = [k for k in next(iter(arrs.values())).files if k.startswith("attr__") and k.endswith("__clean")]
            for k in keys:
                _, model, ex, _ = k.split("__")
                mats = {s: a[k].astype(np.float32) for s, a in arrs.items() if k in a.files}
                for s1, s2 in combinations(sorted(mats), 2):
                    st = stability(mats[s1], mats[s2], list(ks))
                    rows.append({"dataset": ds, "split": int(sp.replace("split", "")), "model": model,
                                 "explainer": ex, "seed_a": s1, "seed_b": s2,
                                 **{kk: float(np.nanmean(v)) for kk, v in st.items()}})
    return pd.DataFrame(rows)


# ------------------------------------------------------------------ main analysis
def analyse(dirs, out):
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    F = load(dirs)
    res = {}
    pred = F["predictive"]; att = F["attacks"]; rec = F["explanations_records"]
    main_split = 0
    # ---- predictive
    p = pred[(pred.split == main_split)]
    cols = ["macro_f1", "balanced_accuracy", "mal_recall", "mal_precision", "ben_precision", "ben_recall",
            "fpr", "fnr", "pr_auc", "roc_auc", "brier", "ece", "threshold"]
    summ = p.groupby(["dataset", "model", "threshold_rule"])[cols].agg(["mean", "std", "count"])
    summ.to_csv(out / "T_predictive_summary.csv")
    res["predictive"] = summ
    # ---- attacks (seed level already)
    a = att[att.split == main_split]
    asumm = a.groupby(["dataset", "model", "attack", "threat", "eps"])[
        ["asr", "asr_at_0p5", "mal_recall_clean", "mal_recall_adv", "mean_linf", "mean_features_changed",
         "syntactic_valid_rate", "threat_model_valid_rate", "onehot_valid_rate", "integer_valid_rate",
         "relation_valid_rate", "asr_syntactic_valid", "asr_threat_model_valid", "mean_queries", "seconds",
         "eligible"]].agg(["mean", "std", "count"])
    asumm.to_csv(out / "T_attacks_summary.csv")
    # ASR CIs
    ci_rows = []
    for k, g in a.groupby(["dataset", "model", "attack", "threat", "eps"]):
        lo, hi = boot_ci(g["asr"])
        ci_rows.append(dict(zip(["dataset", "model", "attack", "threat", "eps"], k), asr_mean=g["asr"].mean(),
                            asr_sd=g["asr"].std(ddof=1), ci_low=lo, ci_high=hi, n=len(g)))
    pd.DataFrame(ci_rows).to_csv(out / "T_attacks_ci.csv", index=False)
    # ---- explanations
    have_expl = len(rec) or len(F["explanations_seed"])
    if have_expl:
        # record-level files when available, otherwise the printed seed-level summaries
        sl = seed_level_explanations(rec) if len(rec) else F["explanations_seed"]
        if len(rec) and len(F["explanations_seed"]):   # runs recovered from a printed log
            have = set(map(tuple, sl[["dataset", "split", "seed"]].values))
            extra = F["explanations_seed"][[tuple(r) not in have for r in F["explanations_seed"][["dataset", "split", "seed"]].values]]
            sl = pd.concat([sl, extra], ignore_index=True)
        sl.to_csv(out / "seed_level_explanations.csv", index=False)
        esumm = sl[sl.split == main_split].groupby(["dataset", "model", "explainer", "view"]).agg(["mean", "std"])
        esumm.to_csv(out / "T_explanations_summary.csv")
        # share of records with negative comprehensiveness: v1-style vs predicted-class definitions
        sl.groupby(["dataset", "model", "explainer", "view"])[["neg_v1", "neg_pred_median", "neg_pred_marginal"]].mean().to_csv(
            out / "T_negative_comprehensiveness_share.csv")
        # ---- primary tests: SHAP vs LIME (paired over seeds)
        tests = []
        emax = EPS_MAIN
        prim_views = ["clean", f"pgd_c_{emax:g}", f"bb_c_{emax:g}", f"pgd_n_{emax:g}", f"bb_n_{emax:g}"]
        outcomes = {"P2_dcomp_marginal": "dcomp_marginal", "dcomp_knn": "dcomp_knn", "dcomp_median": "dcomp_median",
                    "dsuff_marginal": "dsuff_marginal", "P3_stab_spearman": "stab_spearman",
                    "stab_jaccard@3": "stab_jaccard@3", "stab_signed_cosine": "stab_signed_cosine"}
        s0 = sl[sl.split == main_split]
        for (ds, model), g in s0.groupby(["dataset", "model"]):
            for view in g.view.unique():
                if view not in prim_views and not view.startswith(("pgd_", "bb_")):
                    continue
                for oname, col in outcomes.items():
                    if col not in g or g[col].isna().all():
                        continue
                    w = g[g.view == view].pivot_table(index="seed", columns="explainer", values=col)
                    if not {"SHAP", "LIME"} <= set(w.columns):
                        continue
                    t = paired(w["SHAP"] - w["LIME"])
                    tests.append(dict(family=f"{ds}|{model}|{oname}", dataset=ds, model=model, view=view,
                                      outcome=oname, comparison="SHAP-LIME", shap_mean=w["SHAP"].mean(),
                                      lime_mean=w["LIME"].mean(), **t))
                # explanation vs random control (is each explainer better than random?)
                for ex in ["SHAP", "LIME"]:
                    gg = g[(g.view == view) & (g.explainer == ex)]
                    if len(gg):
                        t = paired(gg["dcomp_marginal"])
                        tests.append(dict(family=f"{ds}|{model}|vs_random", dataset=ds, model=model, view=view,
                                          outcome="P2_dcomp_marginal", comparison=f"{ex}-random", **t))
        T = pd.DataFrame(tests)
        if len(T):
            T["p_holm"] = np.nan
            for fam, idx in T.groupby("family").groups.items():
                T.loc[idx, "p_holm"] = holm(T.loc[idx, "p_wilcoxon"].values)
            T.to_csv(out / "T_tests_explainers.csv", index=False)
            res["tests_explainers"] = T
    # ---- condition contrasts (paired over seeds)
    ctr = []
    pv = p[p.threshold_rule == "val_tuned"]
    for ds, g in pv.groupby("dataset"):
        w = g.pivot_table(index="seed", columns="model", values="macro_f1")
        for m in [c for c in w.columns if c != "mlp_clean"]:
            ctr.append(dict(family=f"{ds}|clean_macro_f1", dataset=ds, outcome="clean_macro_f1",
                            comparison=f"{m}-mlp_clean", **paired(w[m] - w["mlp_clean"])))
    emax = EPS_MAIN
    for (ds, threat), g in a[(a.eps == emax)].groupby(["dataset", "threat"]):
        mlp = g[g.attack == "pgd"].pivot_table(index="seed", columns="model", values="asr")
        for m in ["mlp_fgsmat", "mlp_pgdat"]:
            if m in mlp:
                ctr.append(dict(family=f"{ds}|{threat}|ASR_PGD", dataset=ds, outcome=f"ASR_PGD_{threat}",
                                comparison=f"{m}-mlp_clean", **paired(mlp[m] - mlp["mlp_clean"])))
        bb = g[g.attack == "bb"].pivot_table(index="seed", columns="model", values="asr")
        for m in [c for c in bb.columns if c != "mlp_clean"]:
            ctr.append(dict(family=f"{ds}|{threat}|ASR_BB", dataset=ds, outcome=f"ASR_BB_{threat}",
                            comparison=f"{m}-mlp_clean", **paired(bb[m] - bb["mlp_clean"])))
        for mdl in mlp.columns:   # constrained vs unconstrained effect is computed below
            pass
    # threat-model effect
    for (ds, model, attack), g in a[(a.eps == emax) & a.attack.isin(["pgd", "bb"])].groupby(["dataset", "model", "attack"]):
        w = g.pivot_table(index="seed", columns="threat", values="asr")
        for lo, hi in (("constrained", "unconstrained"), ("constrained", "numeric"), ("numeric", "unconstrained")):
            if {lo, hi} <= set(w.columns):
                ctr.append(dict(family=f"{ds}|threat_effect", dataset=ds, outcome=f"ASR_{attack}_{model}",
                                comparison=f"{lo}-{hi}", **paired(w[lo] - w[hi])))
    C = pd.DataFrame(ctr)
    if len(C):
        C["p_holm"] = np.nan
        for fam, idx in C.groupby("family").groups.items():
            C.loc[idx, "p_holm"] = holm(C.loc[idx, "p_wilcoxon"].values)
        C.to_csv(out / "T_tests_conditions.csv", index=False)
    # ---- repeats / cross-seed
    if len(F["repeats"]):
        F["repeats"].groupby(["dataset", "model", "explainer", "source"]).mean(numeric_only=True).to_csv(
            out / "T_repeated_call_stability.csv")
    cs = cross_seed_stability(dirs)
    if not len(cs):   # fall back to printed in-session summaries
        parts = [pd.read_csv(f) for d in dirs for f in Path(d).glob("cross_seed_summary.csv")]
        if parts:
            pd.concat(parts).to_csv(out / "T_cross_seed_stability.csv", index=False)
    if len(cs):
        cs.to_csv(out / "cross_seed_pairs.csv", index=False)
        cs.groupby(["dataset", "split", "model", "explainer"]).mean(numeric_only=True).to_csv(out / "T_cross_seed_stability.csv")
    # ---- split sensitivity (variance components across split x seed)
    if pred.split.nunique() > 1:
        rows = []
        common = sorted(set.intersection(*[set(g.seed) for _, g in pred.groupby("split")]))
        pp = pred[(pred.threshold_rule == "val_tuned") & pred.seed.isin(common)]
        for (ds, m), g in pp.groupby(["dataset", "model"]):
            rows.append(dict(dataset=ds, model=m, outcome="macro_f1", **variance_components(g, "macro_f1")))
        aa = att[(att.eps == emax) & att.seed.isin(common) & att.attack.isin(["pgd", "bb"])]
        for (ds, m, ak, th), g in aa.groupby(["dataset", "model", "attack", "threat"]):
            rows.append(dict(dataset=ds, model=m, outcome=f"asr_{ak}_{th}", **variance_components(g, "asr")))
        if have_expl:
            ss = sl[sl.seed.isin(common)]
            for (ds, m, ex), g in ss[ss.view == "clean"].groupby(["dataset", "model", "explainer"]):
                rows.append(dict(dataset=ds, model=m, outcome=f"dcomp_marginal_{ex}", **variance_components(g, "dcomp_marginal")))
        pd.DataFrame(rows).to_csv(out / "T_split_sensitivity.csv", index=False)
    # ---- power
    n = pred[pred.split == main_split].seed.nunique()
    pw = {"n_seeds": int(n), "mde_dz_alpha05": mde_dz(n, 0.05) if n > 2 else None,
          "mde_dz_alpha05_over_4": mde_dz(n, 0.0125) if n > 2 else None,
          "min_exact_wilcoxon_p_two_sided": 2 / 2 ** n if n else None}
    (out / "power.json").write_text(json.dumps(pw, indent=2))
    # ---- runtime
    if len(F["explanations_runtime"]):
        F["explanations_runtime"].groupby(["dataset", "model", "explainer"])[["ms_per_record"]].agg(
            ["mean", "std"]).to_csv(out / "T_runtime.csv")
    for k, v in F["extras"].items():
        v.to_csv(out / f"extras_{k}.csv", index=False)
    (out / "audit_all.json").write_text(json.dumps(F["audit"], indent=2, default=str))
    figures(F, out, sl if have_expl else None, cs)
    return res


def variance_components(g, col):
    """One-way decomposition: between-split vs within-split (seed) variance of means."""
    by = g.groupby("split")[col]
    means, k = by.mean(), by.size().mean()
    within = by.var(ddof=1).mean()
    between = max(0.0, means.var(ddof=1) - within / k) if len(means) > 1 else np.nan
    return {"grand_mean": g[col].mean(), "split_means": "; ".join(f"{s}:{v:.4f}" for s, v in means.items()),
            "var_between_split": between, "var_within_split_seed": within,
            "share_split": between / (between + within) if (between + within) > 0 else np.nan}


# ------------------------------------------------------------------ figures
def _style():
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.family": "serif", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
                         "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 6.5,
                         "axes.linewidth": 0.6, "lines.linewidth": 1.1, "pdf.fonttype": 42,
                         "axes.spines.top": False, "axes.spines.right": False})
    return plt


COL = {"mlp_clean": "#0072B2", "mlp_fgsmat": "#E69F00", "mlp_pgdat": "#CC79A7", "xgb": "#009E73",
       "SHAP": "#0072B2", "LIME": "#D55E00"}


def figures(F, out, sl, cs):
    plt = _style()
    att = F["attacks"]; att = att[att.split == 0]
    if len(att):
        fig, axes = plt.subplots(1, 2, figsize=(174 / 25.4, 62 / 25.4), sharey=True)
        for ax, ds in zip(axes, ["ton", "edge"]):
            g = att[att.dataset == ds]
            for m in ["mlp_clean", "mlp_fgsmat", "mlp_pgdat", "xgb"]:
                kind = "bb" if m == "xgb" else "pgd"
                for th, ls in (("unconstrained", "-"), ("numeric", ":"), ("constrained", "--")):
                    h = g[(g.model == m) & (g.attack == kind) & (g.threat == th) & (g.eps <= EPS_MAIN + 1e-9)].groupby("eps")["asr"]
                    if not len(h.mean()):
                        continue
                    mu, sd = h.mean(), h.std(ddof=1)
                    ax.errorbar(mu.index, mu.values, yerr=sd.values, ls=ls, marker="o", ms=2.5, capsize=1.5,
                                color=COL[m], label=f"{MODEL_LABEL[m]} ({THREAT_ABBR[th]})")
            ax.set_title(DS_LABEL.get(ds, ds)); ax.set_xlabel(r"$\epsilon$ (normalized $\ell_\infty$)")
            ax.set_ylim(-0.02, 1.02)
        axes[0].set_ylabel("Attack success rate")
        axes[1].legend(ncol=2, frameon=False, loc="upper left", fontsize=5.5)
        fig.tight_layout(); fig.savefig(out / "fig_asr.pdf", bbox_inches="tight"); plt.close(fig)
        # padding budget curve (constrained threat model)
        fig, axes = plt.subplots(1, 2, figsize=(174 / 25.4, 58 / 25.4), sharey=True)
        for ax, ds in zip(axes, ["ton", "edge"]):
            g = att[(att.dataset == ds) & (att.threat == "constrained")]
            for m in ["mlp_clean", "mlp_fgsmat", "mlp_pgdat", "xgb"]:
                kind = "bb" if m == "xgb" else "pgd"
                h = g[(g.model == m) & (g.attack == kind)].groupby("eps")["asr"]
                if len(h.mean()):
                    ax.errorbar(h.mean().index, h.mean().values, yerr=h.std(ddof=1).values, marker="o", ms=2.5,
                                capsize=1.5, color=COL[m], label=MODEL_LABEL[m])
            ax.set_xscale("log"); ax.set_title(DS_LABEL.get(ds, ds))
            ax.set_xlabel(r"padding budget $\epsilon$ (constrained, log scale)")
        axes[0].set_ylabel("Attack success rate"); axes[1].legend(frameon=False)
        fig.tight_layout(); fig.savefig(out / "fig_asr_padding.pdf", bbox_inches="tight"); plt.close(fig)
    if sl is not None and len(sl):
        s0 = sl[(sl.split == 0) & (sl.view == "clean")]
        fig, axes = plt.subplots(1, 2, figsize=(174 / 25.4, 64 / 25.4), sharey=False)
        refs = ["median", "marginal", "knn"]
        for ax, ds in zip(axes, ["ton", "edge"]):
            g = s0[s0.dataset == ds]
            xt, xl = [], []
            x = 0
            for m in ["mlp_clean", "mlp_pgdat", "xgb"]:
                for ri, r in enumerate(refs):
                    for ei, ex in enumerate(["SHAP", "LIME"]):
                        v = g[(g.model == m) & (g.explainer == ex)][f"dcomp_{r}"]
                        if len(v):
                            ax.errorbar(x + ei * 0.3, v.mean(), yerr=v.std(ddof=1), fmt="o", ms=2.5, capsize=1.5,
                                        color=COL[ex], label=ex if (x == 0 and ri == 0) else None)
                    xt.append(x + 0.15); xl.append(f"{MODEL_LABEL[m]}\n{r}")
                    x += 1
                x += 0.5
            ax.axhline(0, color="0.5", lw=0.6)
            ax.set_xticks(xt); ax.set_xticklabels(xl, rotation=90, fontsize=5.5)
            ax.set_title(DS_LABEL.get(ds, ds))
        axes[0].set_ylabel("Comprehensiveness AOPC $-$ random control")
        axes[0].legend(frameon=False)
        fig.tight_layout(); fig.savefig(out / "fig_faithfulness.pdf", bbox_inches="tight"); plt.close(fig)
        # stability vs eps
        sv = sl[(sl.split == 0) & sl.view.str.startswith("pgd_c")].copy()
        sv = sv[sv.view.str.split("_").str[-1].astype(float) <= EPS_MAIN + 1e-9]
        if len(sv):
            sv["eps"] = sv.view.str.split("_").str[-1].astype(float)
            rep = F["repeats"]
            fig, axes = plt.subplots(1, 2, figsize=(174 / 25.4, 60 / 25.4), sharey=True)
            for ax, ds in zip(axes, ["ton", "edge"]):
                g = sv[sv.dataset == ds]
                for m, ls in (("mlp_clean", "-"), ("mlp_pgdat", "--")):
                    for ex in ["SHAP", "LIME"]:
                        h = g[(g.model == m) & (g.explainer == ex)].groupby("eps")["stab_spearman"]
                        if len(h.mean()):
                            ax.errorbar(h.mean().index, h.mean().values, yerr=h.std(ddof=1).values, ls=ls, marker="o",
                                        ms=2.5, capsize=1.5, color=COL[ex], label=f"{ex}, {MODEL_LABEL[m]}")
                if len(rep):
                    r = rep[(rep.dataset == ds) & (rep.model == "mlp_clean")]
                    for ex in ["SHAP", "LIME"]:
                        v = r[r.explainer == ex]["spearman"]
                        if len(v):
                            ax.axhline(v.mean(), color=COL[ex], lw=0.6, ls=":")
                ax.set_title(DS_LABEL.get(ds, ds)); ax.set_xlabel(r"$\epsilon$ (constrained PGD)")
            axes[0].set_ylabel(r"Spearman $\rho$ (clean vs. attacked)")
            axes[1].legend(frameon=False, fontsize=6)
            fig.tight_layout(); fig.savefig(out / "fig_stability.pdf", bbox_inches="tight"); plt.close(fig)
    sw = F["extras"].get("at_sweep")
    if sw is not None and len(sw):
        fig, axes = plt.subplots(1, 2, figsize=(174 / 25.4, 60 / 25.4))
        emax = sw.eps.max()
        for ax, ds in zip(axes, ["ton", "edge"]):
            g = sw[(sw.dataset == ds) & (sw.eps == emax)]
            for th, mk in (("unconstrained", "o"), ("constrained", "s")):
                h = g[g.threat == th].groupby("model")[["clean_macro_f1", "asr"]].mean()
                ax.scatter(h["asr"], h["clean_macro_f1"], marker=mk, s=14, label=th,
                           color="#0072B2" if th[0] == "u" else "#D55E00")
                for name, r in h.iterrows():
                    lab = "clean" if name == "mlp_clean" else name.replace("sweep_", r"$\epsilon_t$=").replace("_", r", $\lambda$=")
                    ax.annotate(lab, (r["asr"], r["clean_macro_f1"]), fontsize=5, xytext=(2, 2), textcoords="offset points")
            ax.set_title(DS_LABEL.get(ds, ds)); ax.set_xlabel(f"PGD ASR at $\\epsilon$={emax:g}")
            ax.set_ylabel("Clean macro-F1 (val-tuned threshold)")
        axes[0].legend(frameon=False)
        fig.tight_layout(); fig.savefig(out / "fig_at_tradeoff.pdf", bbox_inches="tight"); plt.close(fig)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("dirs", nargs="+"); ap.add_argument("--out", required=True)
    a = ap.parse_args()
    analyse(a.dirs, a.out)
    print("analysis written to", a.out)

In [ ]:
%%writefile /kaggle/working/iotxai/report.py
"""Print result tables into the notebook log as delimited CSV blocks, and rebuild
result folders from a pasted log.  This is a fallback for sessions whose output
files are lost: everything the analysis needs at the seed level is printed.

Block format:
    ### BEGIN <run-id> <table>
    <csv>
    ### END <run-id> <table>
"""
from __future__ import annotations

import io
import re
import sys
from pathlib import Path

import pandas as pd

FLOAT_FMT = "%.6g"
ATTACK_COLS = ["dataset", "split", "seed", "model", "attack", "threat", "eps", "eligible", "success", "asr",
               "asr_at_0p5", "mal_recall_clean", "mal_recall_adv", "mean_linf", "mean_features_changed",
               "syntactic_valid_rate", "threat_model_valid_rate", "onehot_valid_rate", "integer_valid_rate",
               "relation_valid_rate", "asr_syntactic_valid", "asr_threat_model_valid", "mean_queries", "seconds",
               "n_attack", "mean_p_drop"]


def print_block(run_id, name, df, out=sys.stdout):
    if df is None or len(df) == 0:
        return
    out.write(f"### BEGIN {run_id} {name}\n")
    df.to_csv(out, index=False, float_format=FLOAT_FMT)
    out.write(f"### END {run_id} {name}\n")
    out.flush()


def run_tables(rdir: Path):
    """Seed-level tables of one finished run (compact)."""
    rdir = Path(rdir)
    t = {}
    for name in ["training", "predictive", "explanations_seed", "explanations_runtime", "repeats"]:
        f = rdir / f"{name}.csv"
        if f.exists():
            t[name] = pd.read_csv(f)
    if (rdir / "repeats.csv").exists():
        r = t["repeats"]
        keys = [c for c in ["dataset", "split", "seed", "model", "explainer", "source"] if c in r]
        t["repeats"] = r.groupby(keys).mean(numeric_only=True).reset_index().drop(columns=["rep_a", "rep_b"], errors="ignore")
    f = rdir / "attacks.csv"
    if f.exists():
        a = pd.read_csv(f)
        t["attacks"] = a[[c for c in ATTACK_COLS if c in a.columns]]
    f = rdir / "attacks_family.csv"
    if f.exists():   # per-family results for the primary attack of each model
        af = pd.read_csv(f)
        main = ((af.attack == "pgd") & af.model.str.startswith("mlp")) | ((af.attack == "bb") & (af.model == "xgb"))
        keep = main & (af.eps.isin([0.05, 1.0]))
        t["attacks_family"] = af[keep]
    f = rdir / "predictive_family.csv"
    if f.exists():
        pf = pd.read_csv(f)
        t["predictive_family"] = pf[pf.threshold_rule == "val_tuned"]
    return t


def print_run(rdir: Path, out=sys.stdout):
    rid = Path(rdir).name
    for name, df in run_tables(rdir).items():
        print_block(rid, name, df, out)


def print_all(results_dir, out=sys.stdout):
    """Reprint every finished run, the data audit and the supplementary experiments."""
    results_dir = Path(results_dir)
    for f in sorted((results_dir / "data_audit").glob("*.json")) if (results_dir / "data_audit").exists() else []:
        if f.name.startswith(("audit_", "split_")):
            out.write(f"### BEGIN data_audit {f.name}\n{f.read_text()}\n### END data_audit {f.name}\n")
    for rdir in sorted((results_dir / "runs").glob("*")):
        if (rdir / "done.json").exists():
            print_run(rdir, out)
    for f in sorted((results_dir / "extras").glob("*.csv")) if (results_dir / "extras").exists() else []:
        print_block("extras", f.stem, pd.read_csv(f), out)
    cs = results_dir / "cross_seed_summary.csv"
    if cs.exists():
        print_block("session", "cross_seed_summary", pd.read_csv(cs), out)


def cross_seed_summary(results_dir):
    """Cross-seed attribution agreement (needs arrays.npz; computed in-session and printed)."""
    from .analysis import cross_seed_stability
    cs = cross_seed_stability([results_dir])
    if len(cs) == 0:
        return cs
    s = cs.groupby(["dataset", "split", "model", "explainer"]).agg(["mean", "std"])
    s.columns = [f"{a}_{b}" for a, b in s.columns]
    s = s.drop(columns=[c for c in s.columns if c.startswith(("seed_a", "seed_b"))]).reset_index()
    s.to_csv(Path(results_dir) / "cross_seed_summary.csv", index=False)
    return s


# ------------------------------------------------------------------ parsing a pasted log
BEGIN = re.compile(r"^### BEGIN (\S+) (\S+)\s*$")


def parse_log(text, out_dir):
    """Rebuild results folders from a printed log. Duplicate blocks (per-run print and
    the final reprint) are written once; later copies overwrite earlier ones."""
    out_dir = Path(out_dir)
    lines = text.splitlines()
    i, n_blocks = 0, 0
    while i < len(lines):
        m = BEGIN.match(lines[i].strip())
        if not m:
            i += 1
            continue
        rid, name = m.group(1), m.group(2)
        end = f"### END {rid} {name}"
        j = i + 1
        body = []
        while j < len(lines) and lines[j].strip() != end:
            body.append(lines[j]); j += 1
        if j >= len(lines):
            print(f"warning: block {rid} {name} not terminated (truncated paste?)")
        if rid == "data_audit":
            d = out_dir / "data_audit"; d.mkdir(parents=True, exist_ok=True)
            (d / name).write_text("\n".join(body))
        elif rid == "extras":
            d = out_dir / "extras"; d.mkdir(parents=True, exist_ok=True)
            pd.read_csv(io.StringIO("\n".join(body))).to_csv(d / f"{name}.csv", index=False)
        elif rid == "session":
            pd.read_csv(io.StringIO("\n".join(body))).to_csv(out_dir / f"{name}.csv", index=False)
        else:
            d = out_dir / "runs" / rid; d.mkdir(parents=True, exist_ok=True)
            pd.read_csv(io.StringIO("\n".join(body))).to_csv(d / f"{name}.csv", index=False)
            (d / "done.json").write_text("{}")
        n_blocks += 1
        i = j + 1
    return n_blocks


if __name__ == "__main__":
    # python -m iotxai.report <pasted_log.txt> <out_dir>
    n = parse_log(Path(sys.argv[1]).read_text(errors="replace"), sys.argv[2])
    print("blocks parsed:", n)

In [ ]:
%%writefile /kaggle/working/iotxai/cli.py
"""Command-line entry point: python -m iotxai.cli --session A --input-root ./data --out ./results"""
import argparse
import time
from dataclasses import asdict
from pathlib import Path

from .config import Config, session_config
from .runner import Logger, run_all
from . import extras


def run_session(session, out, work, input_root="/kaggle/input", ton_csv="", edge_csv="", device="auto"):
    cfg = session_config(session, out_dir=str(out), work_dir=str(work), input_root=str(input_root),
                         ton_csv=ton_csv, edge_csv=edge_csv, device=device)
    log = Logger(Path(out) / f"log_session_{session}.txt")
    t0 = time.time()
    run_all(cfg, log)
    if cfg.run_at_sweep:
        extras.at_sweep(Config(**{**asdict(cfg), "split_ids": [0]}), log)
    if cfg.run_multiclass:
        extras.multiclass(cfg, log)
    if cfg.run_baseline_ablation:
        extras.baseline_ablation(cfg, log)
    if cfg.run_v1_path:
        extras.v1_path(cfg, log)
    if cfg.run_operating_point:
        extras.operating_point(cfg, log)
    if cfg.run_profiling:
        extras.profiling(cfg, log)
    log(f"SESSION {session} finished in {(time.time() - t0) / 3600:.2f} h")
    from .report import cross_seed_summary
    try:
        cross_seed_summary(out)
    except Exception as exc:
        log(f"cross-seed summary failed: {exc}")
    return cfg


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--session", required=True, choices=["TEST", "A", "B", "C1", "C2", "C3", "C4"])
    ap.add_argument("--input-root", default="./data")
    ap.add_argument("--out", default="./results")
    ap.add_argument("--work", default="./cache")
    ap.add_argument("--ton-csv", default="")
    ap.add_argument("--edge-csv", default="")
    ap.add_argument("--device", default="auto")
    a = ap.parse_args()
    run_session(a.session, a.out, a.work, a.input_root, a.ton_csv, a.edge_csv, a.device)

In [ ]:
# ============================ SESSION =====================================
SESSION = "TEST"          # "TEST" first, then "A", "B", "C1", "C2", "C3", "C4" (one notebook version each)
RESUME_FROM = ""          # e.g. "/kaggle/input/<previous-version-output>/results_v2" to skip finished runs
TON_CSV = ""              # leave "" for automatic discovery under /kaggle/input
EDGE_CSV = ""
# ==========================================================================
import sys, shutil, time
from pathlib import Path
sys.path.insert(0, "/kaggle/working")
OUT = "/kaggle/working/results_v2"
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, OUT, dirs_exist_ok=True)
    for d in Path(OUT, "runs").glob("*"):
        if not (d / "done.json").exists():
            shutil.rmtree(d)
from iotxai.cli import run_session
cfg = run_session(SESSION, OUT, "/kaggle/working/cache_v2", "/kaggle/input", TON_CSV, EDGE_CSV)

In [ ]:
# Quick look (also serves as a sanity check for the TEST session)
import pandas as pd
pd.set_option("display.width", 200)
pred = pd.concat([pd.read_csv(f) for f in Path(OUT).glob("runs/*/predictive.csv")])
display(pred[pred.threshold_rule=="val_tuned"].groupby(["dataset","model"])[["macro_f1","mal_recall","ben_precision","fpr","pr_auc"]].mean().round(4))
att = pd.concat([pd.read_csv(f) for f in Path(OUT).glob("runs/*/attacks.csv")])
display(att[att.eps==att.eps.max()].groupby(["dataset","model","attack","threat"])[["asr","syntactic_valid_rate"]].mean().round(3))
for f in sorted(Path(OUT).glob("data_audit/audit_*.json")):
    print(f.name, open(f).read()[:1500])

## Print every result table (copy-paste fallback)
Every run already printed its tables when it finished (blocks starting with `### BEGIN`). This cell reprints
**all** of them. If the output files are ever lost, copy the whole output of this cell (or of the full log) into
a .txt file and send it. The results can be rebuilt from that text with `python -m iotxai.report`.

In [ ]:
from iotxai.report import print_all
print_all(OUT)
with open("/kaggle/working/printed_tables.txt", "w") as fh:
    print_all(OUT, fh)
print("### ALL TABLES PRINTED ###")

In [ ]:
import shutil
from IPython.display import FileLink, display
zip_base = f"/kaggle/working/results_session_{SESSION}"
shutil.make_archive(zip_base, "zip", "/kaggle/working", "results_v2")
print("size MB:", round(Path(zip_base + ".zip").stat().st_size / 2**20, 1))
display(FileLink(f"results_session_{SESSION}.zip"))